In [ ]:
# ============================================================
# NATURAL RESOURCE DEPENDENCE AND MATERIAL EFFICIENCY
# GLOBAL PANEL ANALYSIS, 2000–2021
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

def project_file(filename):
    """Find a project file from the repository root or a notebook subdirectory."""
    candidates = [
        Path.cwd() / filename,
        Path.cwd() / "data" / "processed" / filename,
        Path.cwd() / "data" / "raw" / "unep_irp" / filename,
        Path.cwd().parent / filename,
        Path.cwd().parent / "data" / "processed" / filename,
        Path.cwd().parent / "data" / "raw" / "unep_irp" / filename,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return str(candidate)
    raise FileNotFoundError(
        f"Could not find {filename}. Place it in the documented repository folder "
        "or launch Jupyter from the repository root."
    )


# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Python environment is ready.")
print("Pandas version:", pd.__version__)

In [ ]:
# Load the final cleaned master dataset

file_path = project_file("FINAL_CLEAN_MASTER_2000_2021.csv")

df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

In [ ]:
# Verify the structure
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nYear range:")
print(df["Year"].min(), "to", df["Year"].max())

print("\nNumber of countries/economies:")
print(df["Country Code"].nunique())

print("\nDuplicate country-year observations:")
print(df.duplicated(["Country Code", "Year"]).sum())

In [ ]:
# ============================================================
# STEP 6: MISSING-DATA AUDIT
# ============================================================

# Count missing values for every variable
missing_count = df.isna().sum()

# Calculate percentage missing
missing_percent = (df.isna().mean() * 100).round(2)

missing_table = pd.DataFrame({
    "Missing observations": missing_count,
    "Available observations": len(df) - missing_count,
    "Missing (%)": missing_percent
})

# Display only variables that have missing observations
missing_table[missing_table["Missing observations"] > 0].sort_values(
    "Missing observations",
    ascending=False
)

This one is important because we need to know which countries have missing data, not just how many missing values exist.

In [ ]:
# ============================================================
# STEP 6B: COUNTRY-LEVEL DATA COVERAGE
# ============================================================

variables = [
    "DMC/GDP",
    "GDP/MF",
    "NRR",
    "Oil",
    "Gas",
    "Coal",
    "Mineral",
    "Forest",
    "GDPpc",
    "Industry",
    "Trade",
    "Urban",
    "Renewable",
    "FDI"
]

country_coverage = (
    df.groupby(["Country", "Country Code"])
      .agg(
          Years=("Year", "nunique"),
          **{
              var.replace("/", "_"): (var, lambda x: x.notna().sum())
              for var in variables
          }
      )
      .reset_index()
)

country_coverage.head(20)

This will give us a table showing, for each country, how many observations it has for every variable.

In [ ]:
# ============================================================
# STEP 6C: YEAR COVERAGE
# ============================================================

year_coverage = (
    df.groupby("Year")
      .agg(
          Countries=("Country Code", "nunique"),
          DMC_GDP=("DMC/GDP", "count"),
          GDP_MF=("GDP/MF", "count"),
          NRR=("NRR", "count"),
          GDPpc=("GDPpc", "count"),
          Industry=("Industry", "count"),
          Trade=("Trade", "count"),
          Urban=("Urban", "count"),
          Renewable=("Renewable", "count"),
          FDI=("FDI", "count")
      )
      .reset_index()
)

year_coverage

Why we're doing this

We're checking three different things:

Cell 1: How much data are missing overall?

Cell 2: Which countries are responsible for the missingness?

Cell 3: Are particular years unusually incomplete?

This matters because we don't want to blindly say:

"We have 4,390 observations."

We need to understand how those 4,390 observations are distributed across countries and years before deciding how the econometric sample should be constructed.

In [ ]:
# ============================================================
# STEP 7: FIND COUNTRIES WITH NO DMC/GDP DATA
# ============================================================

dmc_coverage = (
    df.groupby(["Country", "Country Code"])["DMC/GDP"]
      .count()
      .reset_index(name="DMC_GDP_years")
)

countries_no_dmc = dmc_coverage[
    dmc_coverage["DMC_GDP_years"] == 0
].sort_values("Country")

print("Countries with NO DMC/GDP observations:")
print(countries_no_dmc)

print("\nNumber of countries with no DMC/GDP data:",
      len(countries_no_dmc))

Countries with NO DMC/GDP observations: 0

means all 201 country/economies in the master have at least one DMC/GDP observation.

So we do not have four countries with zero DMC/GDP data. The reason the annual table showed 197 in some years is simply that DMC/GDP is not available for those four entities in those particular years, but every entity has DMC/GDP somewhere in the 2000–2021 period.

In [ ]:
# ============================================================
# STEP 8: FIND COUNTRIES WITH INCOMPLETE DMC/GDP
# ============================================================

dmc_incomplete = dmc_coverage[
    dmc_coverage["DMC_GDP_years"] < 22
].sort_values("DMC_GDP_years")

print("Countries with fewer than 22 DMC/GDP observations:")
print(dmc_incomplete.to_string(index=False))

print("\nNumber of countries with incomplete DMC/GDP:",
      len(dmc_incomplete))

I want to investigate those four cases rather than simply accept the missingness.

This will show us exactly which years are present and which years are missing for Sudan, South Sudan, Serbia and Montenegro.

For example, if Serbia's DMC/GDP begins only in 2006, that's a very different data pattern from scattered missing years throughout the period.

We need to know whether these are:

structural gaps at the beginning/end of the series; or
random/intermittent gaps.

That matters for our interpretation of the unbalanced panel.

In [ ]:
# ============================================================
# STEP 8B: SHOW EXACT DMC/GDP YEARS FOR THE 4 INCOMPLETE CASES
# ============================================================

problem_countries = ["SDN", "SSD", "SRB", "MNE"]

for code in problem_countries:
    temp = df[df["Country Code"] == code][
        ["Country", "Country Code", "Year", "DMC/GDP"]
    ].copy()

    print("\n" + "="*60)
    print(temp.to_string(index=False))

we can finally check the original UNEP-IRP data behind those South Sudan zeros.

In [ ]:
# ============================================================
# STEP 8C: CHECK SOUTH SUDAN IN THE ORIGINAL UNEP-IRP FILE
# ============================================================

try:
    unep_path = project_file("mfa_totals_ratios_export.csv")
except FileNotFoundError:
    unep_path = None

if unep_path is None:
    print("Optional UNEP-IRP source export is not included in this repository.")
    print("Source-level validation is skipped; analysis continues with the cleaned master dataset.")
else:
    unep = pd.read_csv(unep_path)
    print("UNEP-IRP file loaded successfully.")
    print("Rows:", unep.shape[0])
    print("Columns:", unep.shape[1])
    print("\nColumn names:")
    print(unep.columns.tolist())


Now we can trace the four problematic DMC cases directly back to the source, rather than guessing whether the zeros/gaps are genuine.

In [ ]:
# ============================================================
# FIND THE DMC/GDP ROW IN THE ORIGINAL UNEP-IRP FILE
# ============================================================

dmc_rows = unep[
    unep["Flow name"].astype(str).str.contains(
        "DMC", case=False, na=False
    )
]

print("Number of rows containing 'DMC':", len(dmc_rows))

print("\nFlow names found:")
print(dmc_rows[["Flow name", "Flow code", "Flow unit"]].drop_duplicates().to_string(index=False))

So our earlier DMC/GDP variable is simply the inverse of this source indicator:

$$ \text{DMC/GDP}=\frac{1}{\text{GDP/DMC}} $$

We should verify the source values directly before making any decision about the four incomplete cases.

In [ ]:
# ============================================================
# CHECK ORIGINAL UNEP-IRP GDP/DMC VALUES
# ============================================================

countries_check = [
    "Sudan",
    "South Sudan",
    "Serbia",
    "Montenegro"
]

dmc_source = unep[
    (unep["Flow code"] == "GDP/DMC") &
    (unep["Country"].isin(countries_check))
].copy()

print("Rows found:", len(dmc_source))

print(
    dmc_source[
        ["Country", "Flow name", "Flow code", "Flow unit"] +
        [str(y) for y in range(2000, 2022)]
    ].to_string(index=False)
)

The original UNEP-IRP source confirms that our cleaned DMC data are behaving correctly.

Most importantly:

South Sudan does NOT have genuine zeros for 2008–2011 in the UNEP source. Those years are NaN in the original source. So the earlier zero values we saw in the cleaned master were introduced somewhere during our extraction/transformation process.

That means we have found a real data-cleaning issue that must be corrected before regression.

So our earlier master needs a correction, specifically for the DMC/GDP variable.

Identify exactly where the false zeros came from

Before fixing the master, let's check the current cleaned file against the original source for these four countries.

In [ ]:
# ============================================================
# COMPARE CLEANED MASTER WITH ORIGINAL UNEP SOURCE
# ============================================================

check = df[
    df["Country"].isin(countries_check)
][["Country", "Country Code", "Year", "DMC/GDP"]].copy()

print(check.to_string(index=False))

The cleaned master df contains the correct inverse values for Montenegro, Serbia, and Sudan. However, South Sudan 2008–2011 are incorrectly recorded as 0.0000.

The original UNEP-IRP source has those four observations as NaN, so they should be missing, not zero.

There is also an important methodological point: we should not simply replace all zeros in DMC/GDP with missing, because we have not established that every zero elsewhere is erroneous. We will correct only the four observations demonstrably inconsistent with the original source.

In [ ]:
# ============================================================
# CORRECT SOUTH SUDAN DMC/GDP VALUES USING ORIGINAL UNEP SOURCE
# ============================================================

# Find South Sudan's original UNEP GDP/DMC row
ssd_unep = unep[
    (unep["Country"] == "South Sudan") &
    (unep["Flow code"] == "GDP/DMC")
].iloc[0]

# Replace the four incorrectly coded zeros with the
# actual source status: missing (NaN)
for year in [2008, 2009, 2010, 2011]:
    df.loc[
        (df["Country Code"] == "SSD") &
        (df["Year"] == year),
        "DMC/GDP"
    ] = pd.NA

print("South Sudan DMC/GDP corrected for 2008–2011.")

print(
    df[
        (df["Country Code"] == "SSD")
    ][["Country", "Country Code", "Year", "DMC/GDP"]]
    .to_string(index=False)
)

Excellent. That correction is now confirmed.

Before we proceed to any regression or sample selection, we need one final DMC integrity check across the entire master. This is important because we found one extraction artefact, and we don't want another hidden zero elsewhere.

What we're checking

A zero in DMC/GDP (kg/USD) would imply zero domestic material consumption per unit of GDP, which is unusual for an economy-wide annual indicator.

We therefore need to establish whether any remaining zeros are:

genuine source values, or
another extraction/conversion artefact like the South Sudan case.

In [ ]:
# ============================================================
# FULL DMC/GDP ZERO-VALUE AUDIT
# ============================================================

dmc_zeros = df[
    df["DMC/GDP"].notna() &
    (df["DMC/GDP"] == 0)
].copy()

print("Number of DMC/GDP observations equal to zero:", len(dmc_zeros))

if len(dmc_zeros) > 0:
    print("\nCountries with DMC/GDP = 0:")
    print(
        dmc_zeros[
            ["Country", "Country Code", "Year", "DMC/GDP"]
        ].to_string(index=False)
    )
else:
    print("\nNo zero values found in DMC/GDP.")

We have 15 zero observations, all for San Marino (SMR), 2000–2014. We should not delete them automatically. We need to establish whether those zeros are actually present in the original UNEP-IRP file.

In [ ]:
# ============================================================
# VERIFY SAN MARINO ZERO VALUES AGAINST ORIGINAL UNEP SOURCE
# ============================================================

smr_unep = unep[
    (unep["Country"] == "San Marino") &
    (unep["Flow code"] == "GDP/DMC")
].copy()

print("Rows found in original UNEP source:", len(smr_unep))

print(
    smr_unep[
        ["Country", "Flow name", "Flow code", "Flow unit"] +
        [str(y) for y in range(2000, 2022)]
    ].to_string(index=False)
)

In [ ]:
# ============================================================
# CORRECT SAN MARINO DMC/GDP VALUES
# ============================================================

df.loc[
    (df["Country Code"] == "SMR") &
    (df["Year"].between(2000, 2014)),
    "DMC/GDP"
] = pd.NA

print("San Marino DMC/GDP corrected for 2000–2014.")

print(
    df[
        df["Country Code"] == "SMR"
    ][["Country", "Country Code", "Year", "DMC/GDP"]]
    .to_string(index=False)
)

Perform a complete source-vs-master DMC validation, rather than checking countries one by one. That will be our proper quality-control test before we proceed to the other variables.

Now let's do the full DMC source-integrity check across the entire master. We want to make sure there are no other discrepancies between our DMC/GDP and the original UNEP GDP/DMC.

In [ ]:
# ============================================================
# FULL DMC SOURCE-INTEGRITY CHECK
# Compare df[DMC/GDP] with the original UNEP GDP/DMC
# ============================================================

# Create a long-format version of the UNEP DMC data
dmc_unep_long = dmc_rows.melt(
    id_vars=["Country", "Flow name", "Flow code", "Flow unit"],
    var_name="Year",
    value_name="GDP_DMC_UNEP"
)

dmc_unep_long["Year"] = pd.to_numeric(
    dmc_unep_long["Year"],
    errors="coerce"
)

# Keep only our study period
dmc_unep_long = dmc_unep_long[
    dmc_unep_long["Year"].between(2000, 2021)
].copy()

# Keep only the DMC flow
dmc_unep_long = dmc_unep_long[
    dmc_unep_long["Flow code"] == "GDP/DMC"
].copy()

# Merge with our cleaned master
dmc_compare = df[
    ["Country", "Country Code", "Year", "DMC/GDP"]
].merge(
    dmc_unep_long[
        ["Country", "Year", "GDP_DMC_UNEP"]
    ],
    on=["Country", "Year"],
    how="left"
)

# Convert UNEP GDP/DMC to DMC/GDP by taking the inverse
dmc_compare["DMC_GDP_expected"] = (
    1 / dmc_compare["GDP_DMC_UNEP"]
)

# Calculate absolute difference
dmc_compare["absolute_difference"] = (
    dmc_compare["DMC/GDP"] -
    dmc_compare["DMC_GDP_expected"]
).abs()

# Identify cases where both values exist but differ
mismatch = dmc_compare[
    dmc_compare["DMC/GDP"].notna() &
    dmc_compare["DMC_GDP_expected"].notna() &
    (dmc_compare["absolute_difference"] > 0.0002)
].copy()

print("============================================================")
print("DMC SOURCE-INTEGRITY CHECK")
print("============================================================")

print("\nMaster observations:", len(dmc_compare))

print(
    "Observations with UNEP GDP/DMC available:",
    dmc_compare["GDP_DMC_UNEP"].notna().sum()
)

print(
    "Observations with cleaned DMC/GDP available:",
    dmc_compare["DMC/GDP"].notna().sum()
)

print(
    "Matching observations:",
    len(dmc_compare) - len(mismatch)
)

print(
    "Potential mismatches:",
    len(mismatch)
)

if len(mismatch) > 0:
    print("\nPotential mismatches:")
    print(
        mismatch[
            [
                "Country",
                "Country Code",
                "Year",
                "DMC/GDP",
                "GDP_DMC_UNEP",
                "DMC_GDP_expected",
                "absolute_difference"
            ]
        ].to_string(index=False)
    )
else:
    print("\nSUCCESS: No material DMC/GDP mismatches found.")

In [ ]:
# ============================================================
# STEP 9: CHECK THE SIZE OF THE DMC DIFFERENCES
# ============================================================

print("Maximum absolute difference:",
      dmc_compare["absolute_difference"].max())

print("\nDifference distribution:")
print(
    dmc_compare["absolute_difference"]
    .dropna()
    .describe()
)

print("\nDifferences within 0.005:",
      (dmc_compare["absolute_difference"] <= 0.005).sum())

print("Differences within 0.01:",
      (dmc_compare["absolute_difference"] <= 0.01).sum())

print("Differences greater than 0.01:",
      (dmc_compare["absolute_difference"] > 0.01).sum())

print("Differences greater than 0.05:",
      (dmc_compare["absolute_difference"] > 0.05).sum())

The median difference is only 0.00007, which is extremely small.

So we should not replace the DMC/GDP values or discard the master. The differences are overwhelmingly consistent with rounding/precision differences between the exported UNEP ratio and the inverse calculation.

However, there are 9 observations above 0.01 that I don't want us to dismiss without checking. This is exactly the kind of forensic check we should do before proceeding to regression.

In [ ]:
# ============================================================
# STEP 10: INSPECT THE 9 LARGEST DMC DIFFERENCES
# ============================================================

large_diff = dmc_compare[
    dmc_compare["absolute_difference"] > 0.01
].copy()

print("Number of observations with difference > 0.01:",
      len(large_diff))

print("\nThese observations are:")
print(
    large_diff[
        [
            "Country",
            "Country Code",
            "Year",
            "DMC/GDP",
            "GDP_DMC_UNEP",
            "DMC_GDP_expected",
            "absolute_difference"
        ]
    ]
    .sort_values("absolute_difference", ascending=False)
    .to_string(index=False)
)

The 9 observations are not obviously erroneous values. They are all very small relative to the magnitude of the DMC/GDP values, and they occur in countries/years where the UNEP-exported GDP/DMC ratio has been rounded to four decimals.

For example, Lesotho 2019:

Master DMC/GDP = 26.9896 kg/USD
UNEP GDP/DMC = 0.0371 USD/kg
Inverting the rounded UNEP value gives 26.9542
Difference = 0.0354 kg/USD
Relative difference ≈ 0.13%

That is not evidence that our master value is wrong. In fact, because the ratio is rounded before inversion, differences of this sort are mathematically expected, especially when the reciprocal is large.

But before we move toward regression, there is one more thing I want to settle: the 19 observations where the master has no DMC/GDP despite being in the 4,390-row country-year structure. This matters because our earlier country audit identified only four countries with incomplete DMC series, so we should reconcile that apparent difference carefully.

In [ ]:
# ============================================================
# STEP 11: IDENTIFY THE 19 DMC OBSERVATIONS WITHOUT UNEP DATA
# ============================================================

dmc_missing_source = dmc_compare[
    dmc_compare["GDP_DMC_UNEP"].isna()
].copy()

print("Number of observations without UNEP GDP/DMC:",
      len(dmc_missing_source))

print("\nObservations:")
print(
    dmc_missing_source[
        [
            "Country",
            "Country Code",
            "Year",
            "DMC/GDP",
            "GDP_DMC_UNEP"
        ]
    ]
    .to_string(index=False)
)

Next: audit the natural-resource-rent variables against their original WDI sources

We need to verify:

Total natural-resource rents
Oil rents
Natural-gas rents
Coal rents
Mineral rents
Forest rents

We'll check source coverage, zeros, missing values, and consistency before doing anything else.

In [ ]:
# ============================================================
# STEP 12: NATURAL-RESOURCE RENT COVERAGE AUDIT
# ============================================================

resource_vars = [
    "NRR",
    "Oil",
    "Gas",
    "Coal",
    "Mineral",
    "Forest"
]

print("NATURAL-RESOURCE RENT COVERAGE")
print("=" * 60)

for var in resource_vars:
    print(f"\n{var}")
    print("-" * 30)

    print("Non-missing:", df[var].notna().sum())
    print("Missing:", df[var].isna().sum())
    print("Zero:", (df[var] == 0).sum())
    print("Negative:", (df[var] < 0).sum())
    print("Minimum:", df[var].min())
    print("Maximum:", df[var].max())

Two important findings

1. The zeros should NOT be converted to missing.

A country can legitimately have zero oil rents, zero coal rents, etc. These are economically meaningful observations. For example, zero oil rents can mean that oil extraction does not generate rents in that country/year.

So we will retain them.

2. Missingness is not catastrophic.

The worst resource variable is gas, with 281 missing observations out of 4,390 (6.4%). Mineral and forest are particularly strong at only 101 missing observations each.

However, there's something we need to investigate before accepting these values: whether the missing observations represent genuine source gaps or whether they were introduced during our merging process.

That's an important distinction for a publication-quality dataset.

In [ ]:
# ============================================================
# STEP 13: COUNTRY-LEVEL RESOURCE-RENT MISSINGNESS
# ============================================================

for var in resource_vars:

    missing_by_country = (
        df[df[var].isna()]
        .groupby(["Country", "Country Code"])
        .size()
        .reset_index(name="Missing_Years")
        .sort_values("Missing_Years", ascending=False)
    )

    print("\n" + "=" * 70)
    print(f"{var} — COUNTRIES WITH MISSING VALUES")
    print("=" * 70)

    print(
        missing_by_country.to_string(index=False)
    )

An important methodological implication

The fact that oil, gas, coal, mineral and forest rents have different coverage is actually one reason our eventual composition model must use complete cases rather than artificially balancing the panel.

It also means we should report the sample transparently. We shouldn't say something like "data were available for 201 countries" and imply that all variables are available for all 201 countries.

Instead, we'll eventually report something like:

The initial master dataset comprised 201 countries/economies observed annually from 2000 to 2021. Variable coverage varied across indicators, and regression samples were therefore determined by complete observations for the variables included in each specification.

In [ ]:
# ============================================================
# STEP 14: GDP PER CAPITA COVERAGE AND VALIDITY AUDIT
# ============================================================

print("GDP PER CAPITA AUDIT")
print("=" * 60)

print("Non-missing:", df["GDPpc"].notna().sum())
print("Missing:", df["GDPpc"].isna().sum())
print("Zero:", (df["GDPpc"] == 0).sum())
print("Negative:", (df["GDPpc"] < 0).sum())
print("Minimum:", df["GDPpc"].min())
print("Maximum:", df["GDPpc"].max())

Find where the 131 missing GDPpc observations occur

In [ ]:
# ============================================================
# STEP 15: COUNTRY-LEVEL GDP PER CAPITA MISSINGNESS
# ============================================================

gdp_missing = (
    df[df["GDPpc"].isna()]
    .groupby(["Country", "Country Code"])
    .size()
    .reset_index(name="Missing_Years")
    .sort_values("Missing_Years", ascending=False)
)

print("Countries with missing GDP per capita:")
print("=" * 60)

print(gdp_missing.to_string(index=False))

What this means for the paper

We should leave GDPpc missing where the source is missing.

We will eventually use complete-case observations for each regression specification. This naturally means the baseline and composition models can have somewhat different samples.

We also should not automatically exclude those 10 economies from the entire master dataset. They may still be useful in specifications where the required variables are available.

Industry value added

Industry is one of our structural controls and has 247 missing observations, so let's inspect its basic validity first.

In [ ]:
# ============================================================
# STEP 16: INDUSTRY VALUE ADDED AUDIT
# ============================================================

print("INDUSTRY VALUE ADDED AUDIT")
print("=" * 60)

print("Non-missing:", df["Industry"].notna().sum())
print("Missing:", df["Industry"].isna().sum())
print("Zero:", (df["Industry"] == 0).sum())
print("Negative:", (df["Industry"] < 0).sum())
print("Minimum:", df["Industry"].min())
print("Maximum:", df["Industry"].max())

The 12 zero observations are not automatically errors. An industry share of zero can occur because of how the underlying national accounts data are reported, so we will retain them for now.

The maximum of 86.67% is high but not inherently impossible for an economy whose production is heavily concentrated in industry. We don't have enough reason at this stage to label it an outlier.

Locate the 247 missing observations

In [ ]:
# ============================================================
# STEP 17: COUNTRY-LEVEL INDUSTRY MISSINGNESS
# ============================================================

industry_missing = (
    df[df["Industry"].isna()]
    .groupby(["Country", "Country Code"])
    .size()
    .reset_index(name="Missing_Years")
    .sort_values("Missing_Years", ascending=False)
)

print("Countries with missing Industry value added:")
print("=" * 60)

print(industry_missing.to_string(index=False))

One thing we'll eventually have to account for is that Industry will reduce the complete-case regression sample, particularly for countries such as Somalia, North Korea, and British Virgin Islands. That's exactly why we're doing the audit before defining the final sample.

Trade is particularly important because it has 683 missing observations, the largest missingness among our main continuous controls.

Let's first check whether the actual values are valid.

In [ ]:
# ============================================================
# STEP 18: TRADE (% GDP) AUDIT
# ============================================================

print("TRADE (% GDP) AUDIT")
print("=" * 60)

print("Non-missing:", df["Trade"].notna().sum())
print("Missing:", df["Trade"].isna().sum())
print("Zero:", (df["Trade"] == 0).sum())
print("Negative:", (df["Trade"] < 0).sum())
print("Minimum:", df["Trade"].min())
print("Maximum:", df["Trade"].max())

The maximum above 100% is not automatically an error. Trade openness is measured as exports + imports as a percentage of GDP, so small, highly open economies can legitimately have values well above 100%.

The 12 zeros are also retained for now.

The main issue is the 683 missing observations (15.6%). This is large enough that Trade may substantially determine our final regression sample, so we'll examine its country-level pattern next.

Trade missingness by country

In [ ]:
# ============================================================
# STEP 19: COUNTRY-LEVEL TRADE MISSINGNESS
# ============================================================

trade_missing = (
    df[df["Trade"].isna()]
    .groupby(["Country", "Country Code"])
    .size()
    .reset_index(name="Missing_Years")
    .sort_values("Missing_Years", ascending=False)
)

print("Countries with missing Trade:")
print("=" * 60)

print(trade_missing.to_string(index=False))

The 22-year gaps are concentrated in particular economies, while others have partial coverage.

A key point is that we should not exclude all of these countries now. For example, a country with 17 missing years may still contribute five valid observations, depending on the eventual specification. The final ≥18-year criterion should be applied to the complete regression sample, not separately to every control.

Also, we should not treat the 437% maximum as an error simply because it is large. Trade/GDP can legitimately exceed 100%.

One thing I want to flag for later

Because Trade has 15.6% missingness, the complete-case baseline sample will probably fall considerably when we combine:

DMC/GDP + NRR + GDPpc + Industry + Trade + Urban + Renewable + FDI.

That's not necessarily a problem. What matters is whether we retain a sufficiently large and geographically diverse sample. We'll calculate that rather than guessing.

In [ ]:
# ============================================================
# STEP 20: URBAN POPULATION AUDIT
# ============================================================

print("URBAN POPULATION (% TOTAL) AUDIT")
print("=" * 60)

print("Non-missing:", df["Urban"].notna().sum())
print("Missing:", df["Urban"].isna().sum())
print("Zero:", (df["Urban"] == 0).sum())
print("Negative:", (df["Urban"] < 0).sum())
print("Minimum:", df["Urban"].min())
print("Maximum:", df["Urban"].max())

The 100% values are possible and do not indicate an error by themselves. We will retain them.

Because Urban has complete coverage, it will not reduce our regression sample at all. We also don't need a separate country-level missingness check for this variable.

In [ ]:
# ============================================================
# STEP 21: RENEWABLE ENERGY CONSUMPTION AUDIT
# ============================================================

print("RENEWABLE ENERGY CONSUMPTION AUDIT")
print("=" * 60)

print("Non-missing:", df["Renewable"].notna().sum())
print("Missing:", df["Renewable"].isna().sum())
print("Zero:", (df["Renewable"] == 0).sum())
print("Negative:", (df["Renewable"] < 0).sum())
print("Minimum:", df["Renewable"].min())
print("Maximum:", df["Renewable"].max())

The 176 zero values are not automatically problematic. Zero renewable energy consumption can be a legitimate observation in the underlying energy statistics. We retain them.

The maximum of 98.3% is also plausible for economies with very high reliance on renewable sources.

So Renewable is unlikely to materially reduce our final sample.

Locate the 28 missing observations

In [ ]:
# ============================================================
# STEP 22: COUNTRY-LEVEL RENEWABLE ENERGY MISSINGNESS
# ============================================================

renewable_missing = (
    df[df["Renewable"].isna()]
    .groupby(["Country", "Country Code"])
    .size()
    .reset_index(name="Missing_Years")
    .sort_values("Missing_Years", ascending=False)
)

print("Countries with missing Renewable energy:")
print("=" * 60)

print(renewable_missing.to_string(index=False))

There are no signs of widespread or random data loss. We will leave these as missing and allow the regression's complete-case construction to handle them.

So Renewable is effectively a low-risk control.

In [ ]:
# ============================================================
# STEP 23: FDI NET INFLOWS AUDIT
# ============================================================

print("FDI NET INFLOWS (% GDP) AUDIT")
print("=" * 60)

print("Non-missing:", df["FDI"].notna().sum())
print("Missing:", df["FDI"].isna().sum())
print("Zero:", (df["FDI"] == 0).sum())
print("Negative:", (df["FDI"] < 0).sum())
print("Minimum:", df["FDI"].min())
print("Maximum:", df["FDI"].max())

The negative values are not automatically errors. FDI net inflows can legitimately be negative because the measure is a net flow: negative values can reflect disinvestment, repayments, or other reversals.

However, values such as −1,303% and +1,710% of GDP are extreme enough that we absolutely need to investigate them before regression.

Identify the extreme FDI observations

In [ ]:
# ============================================================
# STEP 24: INSPECT EXTREME FDI OBSERVATIONS
# ============================================================

fdi_extreme = df[
    ["Country", "Country Code", "Year", "FDI"]
].dropna().copy()

print("20 LOWEST FDI OBSERVATIONS")
print("=" * 70)

print(
    fdi_extreme
    .sort_values("FDI")
    .head(20)
    .to_string(index=False)
)

print("\n\n20 HIGHEST FDI OBSERVATIONS")
print("=" * 70)

print(
    fdi_extreme
    .sort_values("FDI", ascending=False)
    .head(20)
    .to_string(index=False)
)

The extreme values are highly concentrated in very small financial/offshore economies, especially:

Liechtenstein
Cayman Islands
Cyprus
Luxembourg
Aruba

That makes the extreme percentages plausible as ratios, because FDI net inflows divided by a relatively small GDP can produce enormous percentages. So we should not label them as data errors merely because they exceed ±100%.

But they can be highly influential observations in a global panel regression. That's an econometric issue, not necessarily a data-quality issue.

Therefore:

Keep the original FDI observations in the master dataset. Do not winsorize or delete them at this stage.

Later, after estimating the baseline model, we'll conduct an influence/outlier robustness analysis. That is much more defensible than altering the data before seeing whether those observations actually affect our conclusions.

One more important observation

The extremes are not isolated to one accidental value. For example, Liechtenstein and Cayman Islands show repeated very large positive FDI values across multiple years. That strengthens the case that these are characteristics of the reported FDI/GDP series rather than a single corrupted observation.

So FDI is not failed data. It is a valid but heavy-tailed control.

Build the baseline missingness profile

In [ ]:
# ============================================================
# STEP 24: BASELINE MODEL — MISSINGNESS PROFILE
# ============================================================

baseline_vars = [
    "DMC/GDP",
    "NRR",
    "GDPpc",
    "Industry",
    "Trade",
    "Urban",
    "Renewable",
    "FDI"
]

baseline_missing = df[baseline_vars].isna().sum().sort_values(
    ascending=False
)

print("BASELINE MODEL VARIABLES")
print("=" * 60)

print(baseline_missing)

print("\nComplete baseline observations:",
      df[baseline_vars].notna().all(axis=1).sum())

So Trade is clearly the binding constraint in the baseline model. This is not a reason to remove Trade yet; it is something we need to examine carefully because dropping it could substantially increase the sample but would change the model's specification.

Country coverage of the actual baseline sample

Let's see exactly how many usable years each country has after all baseline variables are considered simultaneously.

In [ ]:
# ============================================================
# STEP 25: BASELINE MODEL — COUNTRY COVERAGE
# ============================================================

baseline_complete = df[
    df[baseline_vars].notna().all(axis=1)
].copy()

baseline_country_coverage = (
    baseline_complete
    .groupby(["Country", "Country Code"])
    .size()
    .reset_index(name="Complete_Years")
    .sort_values(
        ["Complete_Years", "Country"],
        ascending=[True, True]
    )
)

print("BASELINE MODEL COUNTRY COVERAGE")
print("=" * 60)

print("Countries with at least 1 complete observation:",
      len(baseline_country_coverage))

print("\nDistribution of complete years per country:")
print(
    baseline_country_coverage["Complete_Years"]
    .describe()
)

print("\nCountries with fewer than 18 complete years:")
print(
    baseline_country_coverage[
        baseline_country_coverage["Complete_Years"] < 18
    ].to_string(index=False)
)

This supports the ≥18-year criterion

The distribution is particularly encouraging because the median country has the full 22 years.

The countries below 18 years are mostly those with known data-availability limitations, rather than a general collapse of the panel. For example, New Caledonia has only 1 complete year, while Afghanistan has 2 and several small/fragile economies have shorter coverage.

I therefore think we should retain ≥18 complete years as our preferred main-sample criterion.

Quantify the baseline sample

Let's now get the exact numbers in a compact table:

All 201 countries/economies in the master.
178 with ≥1 complete baseline observation.
143 with ≥18 complete years.
Total observations in the ≥18-year sample.
Average observations per country.

In [ ]:
# ============================================================
# STEP 26: BASELINE SAMPLE SUMMARY
# ============================================================

eligible_baseline = baseline_country_coverage[
    baseline_country_coverage["Complete_Years"] >= 18
].copy()

print("BASELINE SAMPLE SUMMARY")
print("=" * 60)

print("Countries/economies in master:",
      df["Country Code"].nunique())

print("Countries with >=1 complete baseline observation:",
      len(baseline_country_coverage))

print("Countries with >=18 complete baseline years:",
      len(eligible_baseline))

print("Total complete observations among >=18-year countries:",
      eligible_baseline["Complete_Years"].sum())

print("Mean complete years per eligible country:",
      eligible_baseline["Complete_Years"].mean())

print("Minimum complete years among eligible countries:",
      eligible_baseline["Complete_Years"].min())

print("Maximum complete years among eligible countries:",
      eligible_baseline["Complete_Years"].max())

The most reassuring number is the 21.85-year average. The ≥18-year rule is not forcing us into a highly unbalanced panel: most eligible countries have essentially the entire 2000–2021 period.

But there's an important issue to resolve

Before we officially lock the 143-country baseline sample, we need to know where those countries are geographically.

A data-completeness rule can unintentionally introduce selection bias—for example, if countries with poorer statistical coverage are disproportionately excluded. That doesn't invalidate the study, but we need to understand and report it.

We can do that using the country names/codes already in your dataset.

Inspect the eligible-country list

First, let's simply see the 143 countries that would enter the preferred baseline sample.

In [ ]:
# ============================================================
# STEP 27: LIST COUNTRIES IN THE >=18-YEAR BASELINE SAMPLE
# ============================================================

print("COUNTRIES IN THE >=18-YEAR BASELINE SAMPLE")
print("=" * 70)

print(
    eligible_baseline[
        ["Country", "Country Code", "Complete_Years"]
    ]
    .sort_values("Country")
    .to_string(index=False)
)

Create the baseline eligible dataset

In [ ]:
# ============================================================
# STEP 28: CREATE THE PREFERRED BASELINE SAMPLE
# ============================================================

# Countries meeting the >=18 complete-year criterion
eligible_codes = eligible_baseline["Country Code"].tolist()

# Keep only complete baseline observations
baseline_model = baseline_complete[
    baseline_complete["Country Code"].isin(eligible_codes)
].copy()

print("BASELINE MODEL DATASET")
print("=" * 60)

print("Rows:", len(baseline_model))
print("Countries/economies:",
      baseline_model["Country Code"].nunique())

print("Years:",
      baseline_model["Year"].min(),
      "to",
      baseline_model["Year"].max())

print("\nDuplicate country-year observations:",
      baseline_model.duplicated(
          ["Country Code", "Year"]
      ).sum())

print("\nMissing values in model variables:")
print(
    baseline_model[baseline_vars]
    .isna()
    .sum()
)

In [ ]:
# ============================================================
# STEP 29: CHECK EXISTING COLUMNS FOR REGION INFORMATION
# ============================================================

print("Columns in baseline_model:")
print(baseline_model.columns.tolist())

In [ ]:
# ============================================================
# STEP 30: IDENTIFY ECONOMIES THAT MAY NEED SPECIAL HANDLING
# ============================================================

unique_countries = (
    baseline_model[
        ["Country", "Country Code"]
    ]
    .drop_duplicates()
    .sort_values("Country")
)

print("ALL 143 COUNTRIES/ECONOMIES")
print("=" * 70)

print(unique_countries.to_string(index=False))

We first need to examine the distributions of the variables. In particular:

DMC/GDP is likely right-skewed.
Resource rents are bounded at zero but can be highly skewed.
GDP per capita is strongly right-skewed.
Trade and FDI can have extreme values.
Industry may have substantial between-country variation.

This will determine whether our proposed transformations (ln(DMC/GDP), asinh(resource rents), ln(GDPpc)) are sensible.

In [ ]:
# ============================================================
# STEP 31: BASELINE SAMPLE DESCRIPTIVE STATISTICS
# ============================================================

baseline_numeric = [
    "DMC/GDP",
    "NRR",
    "GDPpc",
    "Industry",
    "Trade",
    "Urban",
    "Renewable",
    "FDI"
]

descriptive_stats = baseline_model[baseline_numeric].describe().T

descriptive_stats["skewness"] = (
    baseline_model[baseline_numeric]
    .skew()
)

print("BASELINE SAMPLE DESCRIPTIVE STATISTICS")
print("=" * 80)

print(
    descriptive_stats[
        [
            "count",
            "mean",
            "std",
            "min",
            "25%",
            "50%",
            "75%",
            "max",
            "skewness"
        ]
    ].round(4)
)

We previously saw positive DMC/GDP values when examining the missing countries, but we have now discovered:

DMC/GDP minimum = −8.1537 kg/USD

Domestic material consumption per unit GDP should not ordinarily be negative if it is truly a physical material-intensity measure. So we must determine exactly which observations are negative and whether they represent a data/construction problem.

Investigate negative DMC/GDP observations

In [ ]:
# ============================================================
# STEP 32: INVESTIGATE NEGATIVE DMC/GDP VALUES
# ============================================================

negative_dmc = baseline_model[
    baseline_model["DMC/GDP"] < 0
].copy()

print("NEGATIVE DMC/GDP OBSERVATIONS")
print("=" * 70)

print("Number of negative observations:",
      len(negative_dmc))

print("\nNegative observations:")
print(
    negative_dmc[
        [
            "Country",
            "Country Code",
            "Year",
            "DMC/GDP",
            "GDP/MF"
        ]
    ]
    .sort_values(["Country", "Year"])
    .to_string(index=False)
)

And there is an especially interesting clue:

Brunei: −0.1047, −0.0996
Mozambique: −1.6125, −8.1537

These aren't tiny floating-point negatives such as -0.0000001. They are materially negative.

So we need to trace these four observations directly back to the original UNEP-IRP export.

Inspect the original UNEP rows

In [ ]:
# ============================================================
# STEP 32A: TRACE NEGATIVE DMC VALUES TO ORIGINAL UNEP FILE
# ============================================================

print("UNEP rows for Brunei Darussalam and Mozambique")
print("=" * 70)

source_check = unep[
    unep["Country"].isin([
        "Brunei Darussalam",
        "Mozambique"
    ])
].copy()

print(
    source_check[
        [
            "Country",
            "Flow name",
            "Flow code",
            "Flow unit"
        ]
        + [str(y) for y in range(2015, 2022)]
    ].to_string(index=False)
)

In [ ]:
# ============================================================
# STEP 32B: LOAD FINAL MASTER AND FIND ALL NON-POSITIVE DMC/GDP
# ============================================================

import pandas as pd

master = pd.read_csv(project_file("FINAL_CLEAN_MASTER_2000_2021.csv"))

dmc_nonpositive = master[
    master["DMC/GDP"] <= 0
].copy()

print("Total DMC/GDP observations <= 0:", len(dmc_nonpositive))
print("Countries affected:", dmc_nonpositive["Country"].nunique())

print("\nAll non-positive DMC/GDP observations:")
print(
    dmc_nonpositive[
        ["Country", "Country Code", "Year", "DMC/GDP", "GDP/MF"]
    ]
    .sort_values(["Country", "Year"])
    .to_string(index=False)
)

In [ ]:
# ============================================================
# STEP 32C: CHECK NON-POSITIVE DMC OBSERVATIONS IN BASELINE
# SAMPLE
# ============================================================

baseline_codes = set(baseline_model["Country Code"])

problem_baseline = master[
    (master["Country Code"].isin(baseline_codes)) &
    (master["DMC/GDP"] <= 0)
].copy()

print("Non-positive DMC/GDP observations in baseline sample:",
      len(problem_baseline))

print("\nCountries affected:",
      problem_baseline["Country"].nunique())

print("\nObservations:")
print(
    problem_baseline[
        ["Country", "Country Code", "Year", "DMC/GDP", "GDP/MF"]
    ]
    .sort_values(["Country", "Year"])
    .to_string(index=False)
)

Primary DMC/GDP analysis

Use strictly positive DMC/GDP observations only.

That means removing the four negative observations from models where DMC/GDP is the dependent variable.

This is not arbitrary data cleaning. It is because:

DMC/GDP is intended to measure material intensity.
Material intensity cannot meaningfully be interpreted on a logarithmic scale when negative.
The four negative values are generated by unusual physical trade balances in the underlying material-flow accounting.
We have verified that they originate in the official UNEP-IRP dataset itself.
Removing them does not remove entire countries:
Brunei remains in the panel.
Mozambique remains in the panel.
We are removing four country-year observations, not excluding Brunei or Mozambique altogether.

In [ ]:
# ============================================================
# STEP 32D: QUANTIFY EFFECT OF REMOVING NEGATIVE DMC/GDP
# ============================================================

baseline_positive = baseline_model[
    baseline_model["DMC/GDP"] > 0
].copy()

print("Original baseline observations:", len(baseline_model))
print("Positive DMC/GDP observations:", len(baseline_positive))

print("\nOriginal countries:", baseline_model["Country Code"].nunique())
print("Countries after removing negative DMC/GDP:",
      baseline_positive["Country Code"].nunique())

print("\nObservations removed:",
      len(baseline_model) - len(baseline_positive))

print("\nCountries affected:")
print(
    baseline_model[
        baseline_model["DMC/GDP"] <= 0
    ][["Country", "Country Code", "Year", "DMC/GDP"]]
    .sort_values(["Country", "Year"])
    .to_string(index=False)
)

The next thing I want to check is the distribution of the positive DMC/GDP variable.

We already saw substantial right skew:

Mean = 2.70
Median = 1.98
SD = 2.53
Skewness = 1.92
Maximum = 20.51

That strongly suggests a log transformation may be appropriate, but we should verify the distribution rather than decide from skewness alone.

So let's now create the positive-DMC dataset in memory only and examine its distribution.

Run this:

In [ ]:
# ============================================================
# STEP 32E: DMC/GDP DISTRIBUTION AFTER POSITIVE-VALUE SCREEN
# ============================================================

baseline_positive = baseline_model[
    baseline_model["DMC/GDP"] > 0
].copy()

dmc = baseline_positive["DMC/GDP"]

print("DMC/GDP distribution after screening")
print("=" * 60)

print("N:", dmc.count())
print("Mean:", dmc.mean())
print("Median:", dmc.median())
print("Std. Dev.:", dmc.std())
print("Minimum:", dmc.min())
print("Maximum:", dmc.max())
print("Skewness:", dmc.skew())
print("25th percentile:", dmc.quantile(0.25))
print("75th percentile:", dmc.quantile(0.75))
print("90th percentile:", dmc.quantile(0.90))
print("95th percentile:", dmc.quantile(0.95))
print("99th percentile:", dmc.quantile(0.99))

We're not committing to the log transformation solely because the raw variable is skewed. This comparison will give us an empirical basis for the decision.

In [ ]:
# ============================================================
# STEP 32F: COMPARE RAW AND LOG DMC/GDP DISTRIBUTIONS
# ============================================================

import numpy as np

dmc_raw = baseline_positive["DMC/GDP"]
dmc_log = np.log(dmc_raw)

print("RAW DMC/GDP")
print("=" * 60)
print("Mean:", dmc_raw.mean())
print("Median:", dmc_raw.median())
print("Std. Dev.:", dmc_raw.std())
print("Skewness:", dmc_raw.skew())

print("\nLOG(DMC/GDP)")
print("=" * 60)
print("Mean:", dmc_log.mean())
print("Median:", dmc_log.median())
print("Std. Dev.:", dmc_log.std())
print("Skewness:", dmc_log.skew())
print("Minimum:", dmc_log.min())
print("Maximum:", dmc_log.max())

In [ ]:
# ============================================================
# STEP 32G: NRR DISTRIBUTION IN FINAL DMC SAMPLE
# ============================================================

nrr = baseline_positive["NRR"]

print("NRR DISTRIBUTION")
print("=" * 60)

print("N:", nrr.count())
print("Mean:", nrr.mean())
print("Median:", nrr.median())
print("Std. Dev.:", nrr.std())
print("Minimum:", nrr.min())
print("Maximum:", nrr.max())
print("Skewness:", nrr.skew())

print("\nPercentiles")
print("-" * 60)
print("25th:", nrr.quantile(0.25))
print("50th:", nrr.quantile(0.50))
print("75th:", nrr.quantile(0.75))
print("90th:", nrr.quantile(0.90))
print("95th:", nrr.quantile(0.95))
print("99th:", nrr.quantile(0.99))

print("\nZero observations:", (nrr == 0).sum())
print("Positive observations:", (nrr > 0).sum())

In [ ]:
# ============================================================
# STEP 32H: COMPARE RAW NRR AND ASINH(NRR)
# ============================================================

import numpy as np

nrr_raw = baseline_positive["NRR"]
nrr_asinh = np.arcsinh(nrr_raw)

print("RAW NRR")
print("=" * 60)
print("Mean:", nrr_raw.mean())
print("Median:", nrr_raw.median())
print("Std. Dev.:", nrr_raw.std())
print("Skewness:", nrr_raw.skew())

print("\nASINH(NRR)")
print("=" * 60)
print("Mean:", nrr_asinh.mean())
print("Median:", np.median(nrr_asinh))
print("Std. Dev.:", nrr_asinh.std())
print("Skewness:", pd.Series(nrr_asinh).skew())
print("Minimum:", nrr_asinh.min())
print("Maximum:", nrr_asinh.max())

print("\nZero observations preserved:",
      (nrr_asinh == 0).sum())

In [ ]:
# ============================================================
# STEP 32I: GDP PER CAPITA DISTRIBUTION
# ============================================================

gdp_pc = baseline_positive["GDPpc"]

print("GDP PER CAPITA DISTRIBUTION")
print("=" * 60)

print("N:", gdp_pc.count())
print("Mean:", gdp_pc.mean())
print("Median:", gdp_pc.median())
print("Std. Dev.:", gdp_pc.std())
print("Minimum:", gdp_pc.min())
print("Maximum:", gdp_pc.max())
print("Skewness:", gdp_pc.skew())

print("\nPercentiles")
print("-" * 60)
print("25th:", gdp_pc.quantile(0.25))
print("50th:", gdp_pc.quantile(0.50))
print("75th:", gdp_pc.quantile(0.75))
print("90th:", gdp_pc.quantile(0.90))
print("95th:", gdp_pc.quantile(0.95))
print("99th:", gdp_pc.quantile(0.99))

In [ ]:
# ============================================================
# STEP 32J: COMPARE RAW GDP PER CAPITA AND LOG GDP PER CAPITA
# ============================================================

gdp_pc_raw = baseline_positive["GDPpc"]
gdp_pc_log = np.log(gdp_pc_raw)

print("RAW GDP PER CAPITA")
print("=" * 60)
print("Mean:", gdp_pc_raw.mean())
print("Median:", gdp_pc_raw.median())
print("Std. Dev.:", gdp_pc_raw.std())
print("Skewness:", gdp_pc_raw.skew())

print("\nLOG(GDP PER CAPITA)")
print("=" * 60)
print("Mean:", gdp_pc_log.mean())
print("Median:", gdp_pc_log.median())
print("Std. Dev.:", gdp_pc_log.std())
print("Skewness:", gdp_pc_log.skew())
print("Minimum:", gdp_pc_log.min())
print("Maximum:", gdp_pc_log.max())

In [ ]:
# ============================================================
# STEP 32K: INDUSTRY VALUE ADDED DISTRIBUTION
# ============================================================

industry = baseline_positive["Industry"]

print("INDUSTRY VALUE ADDED (% GDP)")
print("=" * 60)

print("N:", industry.count())
print("Mean:", industry.mean())
print("Median:", industry.median())
print("Std. Dev.:", industry.std())
print("Minimum:", industry.min())
print("Maximum:", industry.max())
print("Skewness:", industry.skew())

print("\nPercentiles")
print("-" * 60)
print("25th:", industry.quantile(0.25))
print("50th:", industry.quantile(0.50))
print("75th:", industry.quantile(0.75))
print("90th:", industry.quantile(0.90))
print("95th:", industry.quantile(0.95))
print("99th:", industry.quantile(0.99))

In [ ]:
# ============================================================
# STEP 32L: TRADE DISTRIBUTION
# ============================================================

trade = baseline_positive["Trade"]

print("TRADE (% GDP)")
print("=" * 60)

print("N:", trade.count())
print("Mean:", trade.mean())
print("Median:", trade.median())
print("Std. Dev.:", trade.std())
print("Minimum:", trade.min())
print("Maximum:", trade.max())
print("Skewness:", trade.skew())

print("\nPercentiles")
print("-" * 60)
print("25th:", trade.quantile(0.25))
print("50th:", trade.quantile(0.50))
print("75th:", trade.quantile(0.75))
print("90th:", trade.quantile(0.90))
print("95th:", trade.quantile(0.95))
print("99th:", trade.quantile(0.99))

print("\nZero observations:", (trade == 0).sum())

In [ ]:
# ============================================================
# STEP 32M: COMPARE RAW TRADE AND LOG TRADE
# ============================================================

trade_raw = baseline_positive["Trade"]
trade_log = np.log(trade_raw)

print("RAW TRADE (% GDP)")
print("=" * 60)
print("Mean:", trade_raw.mean())
print("Median:", trade_raw.median())
print("Std. Dev.:", trade_raw.std())
print("Skewness:", trade_raw.skew())

print("\nLOG(TRADE)")
print("=" * 60)
print("Mean:", trade_log.mean())
print("Median:", trade_log.median())
print("Std. Dev.:", trade_log.std())
print("Skewness:", trade_log.skew())
print("Minimum:", trade_log.min())
print("Maximum:", trade_log.max())

In [ ]:
# ============================================================
# STEP 32N: URBAN POPULATION DISTRIBUTION
# ============================================================

urban = baseline_positive["Urban"]

print("URBAN POPULATION (% TOTAL)")
print("=" * 60)

print("N:", urban.count())
print("Mean:", urban.mean())
print("Median:", urban.median())
print("Std. Dev.:", urban.std())
print("Minimum:", urban.min())
print("Maximum:", urban.max())
print("Skewness:", urban.skew())

print("\nPercentiles")
print("-" * 60)
print("25th:", urban.quantile(0.25))
print("50th:", urban.quantile(0.50))
print("75th:", urban.quantile(0.75))
print("90th:", urban.quantile(0.90))
print("95th:", urban.quantile(0.95))
print("99th:", urban.quantile(0.99))

In [ ]:
# ============================================================
# STEP 32O: RENEWABLE ENERGY CONSUMPTION DISTRIBUTION
# ============================================================

renewable = baseline_positive["Renewable"]

print("RENEWABLE ENERGY CONSUMPTION (% FINAL ENERGY)")
print("=" * 60)

print("N:", renewable.count())
print("Mean:", renewable.mean())
print("Median:", renewable.median())
print("Std. Dev.:", renewable.std())
print("Minimum:", renewable.min())
print("Maximum:", renewable.max())
print("Skewness:", renewable.skew())

print("\nPercentiles")
print("-" * 60)
print("25th:", renewable.quantile(0.25))
print("50th:", renewable.quantile(0.50))
print("75th:", renewable.quantile(0.75))
print("90th:", renewable.quantile(0.90))
print("95th:", renewable.quantile(0.95))
print("99th:", renewable.quantile(0.99))

print("\nZero observations:", (renewable == 0).sum())

In [ ]:
# ============================================================
# STEP 32P: FDI DISTRIBUTION
# ============================================================

fdi = baseline_positive["FDI"]

print("FOREIGN DIRECT INVESTMENT, NET INFLOWS (% GDP)")
print("=" * 60)

print("N:", fdi.count())
print("Mean:", fdi.mean())
print("Median:", fdi.median())
print("Std. Dev.:", fdi.std())
print("Minimum:", fdi.min())
print("Maximum:", fdi.max())
print("Skewness:", fdi.skew())

print("\nPercentiles")
print("-" * 60)
print("1st:", fdi.quantile(0.01))
print("5th:", fdi.quantile(0.05))
print("25th:", fdi.quantile(0.25))
print("50th:", fdi.quantile(0.50))
print("75th:", fdi.quantile(0.75))
print("90th:", fdi.quantile(0.90))
print("95th:", fdi.quantile(0.95))
print("99th:", fdi.quantile(0.99))

print("\nNegative observations:", (fdi < 0).sum())
print("Zero observations:", (fdi == 0).sum())
print("Positive observations:", (fdi > 0).sum())

In [ ]:
# ============================================================
# STEP 32Q: CREATE TRANSFORMED BASELINE MODELLING DATASET
# ============================================================

model = baseline_positive.copy()

# Transformations justified by the distributional diagnostics
model["ln_DMC_GDP"] = np.log(model["DMC/GDP"])
model["asinh_NRR"] = np.arcsinh(model["NRR"])
model["ln_GDPpc"] = np.log(model["GDPpc"])
model["ln_Trade"] = np.log(model["Trade"])

# Variables retained in levels
model["Industry_level"] = model["Industry"]
model["Urban_level"] = model["Urban"]
model["Renewable_level"] = model["Renewable"]
model["FDI_level"] = model["FDI"]

print("TRANSFORMED BASELINE DATASET")
print("=" * 60)

print("Observations:", len(model))
print("Countries:", model["Country Code"].nunique())
print("Years:", model["Year"].min(), "to", model["Year"].max())

print("\nMissing values in modelling variables:")
print(
    model[
        [
            "ln_DMC_GDP",
            "asinh_NRR",
            "ln_GDPpc",
            "ln_Trade",
            "Industry_level",
            "Urban_level",
            "Renewable_level",
            "FDI_level"
        ]
    ].isna().sum()
)

Multicollinearity

Before we estimate anything, we need to determine whether the explanatory variables contain problematic linear relationships.

We'll start with pairwise correlations. This is easier to interpret than jumping immediately to VIF.

In [ ]:
# ============================================================
# STEP 32R: PAIRWISE CORRELATIONS
# ============================================================

corr_vars = [
    "ln_DMC_GDP",
    "asinh_NRR",
    "ln_GDPpc",
    "Industry_level",
    "ln_Trade",
    "Urban_level",
    "Renewable_level",
    "FDI_level"
]

corr_matrix = model[corr_vars].corr()

print("PAIRWISE CORRELATION MATRIX")
print("=" * 80)

print(corr_matrix.round(3).to_string())

Now run VIF

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd

vif_vars = [
    "asinh_NRR",
    "ln_GDPpc",
    "Industry_level",
    "ln_Trade",
    "Urban_level",
    "Renewable_level",
    "FDI_level"
]

X_vif = model[vif_vars].copy()
X_vif = X_vif.astype(float)

vif_table = pd.DataFrame({
    "Variable": X_vif.columns,
    "VIF": [
        variance_inflation_factor(X_vif.values, i)
        for i in range(X_vif.shape[1])
    ]
})

print("="*80)
print("VARIANCE INFLATION FACTOR (VIF)")
print("="*80)
print(vif_table.sort_values("VIF", ascending=False).round(3).to_string(index=False))

Within-country variation and FE-relevant correlations

In [ ]:
# ============================================================
# STEP 33 — WITHIN-COUNTRY VARIATION AND CORRELATION
# ============================================================

fe_vars = [
    "ln_DMC_GDP",
    "asinh_NRR",
    "ln_GDPpc",
    "Industry_level",
    "ln_Trade",
    "Urban_level",
    "Renewable_level",
    "FDI_level"
]

# Country-demean each variable
within = model[["Country"] + fe_vars].copy()

for v in fe_vars:
    within[v + "_within"] = (
        within[v] - within.groupby("Country")[v].transform("mean")
    )

within_vars = [v + "_within" for v in fe_vars]

within_corr = within[within_vars].corr()

print("="*90)
print("WITHIN-COUNTRY CORRELATION MATRIX")
print("="*90)
print(within_corr.round(3).to_string())

Calculate FE-relevant VIF

In [ ]:
# ============================================================
# STEP 34 — WITHIN-COUNTRY / FE-RELEVANT VIF
# ============================================================

from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd

x_vars = [
    "asinh_NRR",
    "ln_GDPpc",
    "Industry_level",
    "ln_Trade",
    "Urban_level",
    "Renewable_level",
    "FDI_level"
]

# Country-demean explanatory variables
X_within = model[["Country"] + x_vars].copy()

for v in x_vars:
    X_within[v + "_within"] = (
        X_within[v]
        - X_within.groupby("Country")[v].transform("mean")
    )

X_fe = X_within[[v + "_within" for v in x_vars]].copy()

# Drop variables with zero variance, if any
X_fe = X_fe.loc[:, X_fe.std() > 0]

fe_vif = pd.DataFrame({
    "Variable": X_fe.columns.str.replace("_within", "", regex=False),
    "FE_VIF": [
        variance_inflation_factor(X_fe.values, i)
        for i in range(X_fe.shape[1])
    ]
})

print("="*80)
print("FE-RELEVANT VIF — COUNTRY-DEMEANED EXPLANATORY VARIABLES")
print("="*80)
print(
    fe_vif
    .sort_values("FE_VIF", ascending=False)
    .round(3)
    .to_string(index=False)
)

Now we move into the genuinely important panel-data diagnostics.

We need to determine whether the residual structure supports our eventual inference.

In particular, we need to investigate:

Cross-sectional dependence
Heteroskedasticity
Serial correlation
Slope heterogeneity
Potential stationarity concerns

These diagnostics will help determine how we estimate the model and, particularly, how we obtain reliable standard errors.

Step 35 — Set up the panel structure

Before running the tests, let's formally establish the panel dimensions and within-country variation.

In [ ]:
# ============================================================
# STEP 35 — PANEL STRUCTURE
# ============================================================

panel = model.copy()

panel = panel.sort_values(["Country", "Year"]).reset_index(drop=True)

print("="*80)
print("PANEL STRUCTURE")
print("="*80)

print(f"Countries: {panel['Country'].nunique()}")
print(f"Observations: {len(panel):,}")
print(f"Years: {panel['Year'].min()}–{panel['Year'].max()}")

obs_by_country = panel.groupby("Country").size()

print(f"Minimum observations per country: {obs_by_country.min()}")
print(f"Maximum observations per country: {obs_by_country.max()}")
print(f"Mean observations per country: {obs_by_country.mean():.2f}")
print(f"Median observations per country: {obs_by_country.median():.2f}")

print("\nCountries by number of observations:")
print(obs_by_country.value_counts().sort_index().to_string())

Now we need the within-variation output

In [ ]:
variation_vars = [
    "ln_DMC_GDP",
    "asinh_NRR",
    "ln_GDPpc",
    "Industry_level",
    "ln_Trade",
    "Urban_level",
    "Renewable_level",
    "FDI_level"
]

variation_rows = []

for v in variation_vars:
    country_means = panel.groupby("Country")[v].transform("mean")
    within_component = panel[v] - country_means

    variation_rows.append({
        "Variable": v,
        "Pooled_SD": panel[v].std(),
        "Within_SD": within_component.std(),
        "Within_to_Pooled": within_component.std() / panel[v].std(),
        "Min_within_SD_country": panel.groupby("Country")[v].std().min(),
        "Median_within_SD_country": panel.groupby("Country")[v].std().median(),
        "Max_within_SD_country": panel.groupby("Country")[v].std().max()
    })

variation_df = pd.DataFrame(variation_rows)

print("="*100)
print("WITHIN-COUNTRY VARIATION")
print("="*100)
print(
    variation_df
    .round(4)
    .to_string(index=False)
)

Cross-sectional dependence

In [ ]:
# ============================================================
# STEP 36 — PRELIMINARY TWO-WAY FIXED-EFFECTS MODEL
# FOR CROSS-SECTIONAL DEPENDENCE DIAGNOSTICS
# ============================================================

import pandas as pd
import numpy as np

# Check whether linearmodels is installed
try:
    from linearmodels.panel import PanelOLS
    print("linearmodels is available.")
except ImportError:
    print("linearmodels is NOT installed.")
    print("Run: !pip install linearmodels")

In [ ]:
!pip install -q linearmodels

In [ ]:
from linearmodels.panel import PanelOLS

print("linearmodels installed successfully.")

In [ ]:
# ============================================================
# STEP 36C — PRELIMINARY TWO-WAY FIXED-EFFECTS MODEL
# ============================================================

from linearmodels.panel import PanelOLS

fe_data = model.copy()
fe_data = fe_data.set_index(["Country", "Year"])

y = fe_data["ln_DMC_GDP"]

X = fe_data[
    [
        "asinh_NRR",
        "ln_GDPpc",
        "Industry_level",
        "ln_Trade",
        "Urban_level",
        "Renewable_level",
        "FDI_level"
    ]
]

prelim_fe = PanelOLS(
    y,
    X,
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)

prelim_result = prelim_fe.fit()

print(prelim_result.summary)

In [ ]:
# ============================================================
# STEP 36D — EXTRACT TWO-WAY FE RESIDUALS
# ============================================================

residuals = prelim_result.resids.copy()

residual_df = residuals.reset_index()
residual_df.columns = ["Country", "Year", "Residual"]

print("="*80)
print("TWO-WAY FE RESIDUALS")
print("="*80)

print(f"Residual observations: {len(residual_df):,}")
print(f"Countries: {residual_df['Country'].nunique()}")
print(f"Years: {residual_df['Year'].nunique()}")

print("\nResidual summary:")
print(residual_df["Residual"].describe().round(4))

Cross-sectional dependence

In [ ]:
# ============================================================
# STEP 37 — PESARAN CROSS-SECTIONAL DEPENDENCE TEST
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import norm

# residual_df already contains:
# Country, Year, Residual

resid_wide = residual_df.pivot(
    index="Year",
    columns="Country",
    values="Residual"
)

countries = resid_wide.columns.tolist()

pair_correlations = []
pair_counts = []

for i in range(len(countries)):
    for j in range(i + 1, len(countries)):

        pair = resid_wide[[countries[i], countries[j]]].dropna()

        # Need at least 3 overlapping observations
        if len(pair) >= 3:
            r = pair.iloc[:, 0].corr(pair.iloc[:, 1])

            if pd.notna(r):
                pair_correlations.append(r)
                pair_counts.append(len(pair))

pair_correlations = np.array(pair_correlations)
pair_counts = np.array(pair_counts)

N = len(countries)
T = residual_df["Year"].nunique()

# Pesaran CD statistic
CD = np.sqrt(
    2 * T / (N * (N - 1))
) * np.sum(pair_correlations)

p_value = 2 * (1 - norm.cdf(abs(CD)))

print("="*80)
print("PESARAN CROSS-SECTIONAL DEPENDENCE TEST")
print("="*80)

print(f"Number of countries: {N}")
print(f"Number of years: {T}")
print(f"Country pairs used: {len(pair_correlations):,}")
print(f"Mean pairwise residual correlation: {pair_correlations.mean():.4f}")
print(f"Median pairwise residual correlation: {np.median(pair_correlations):.4f}")
print(f"Minimum pairwise correlation: {pair_correlations.min():.4f}")
print(f"Maximum pairwise correlation: {pair_correlations.max():.4f}")
print(f"Mean overlapping years per pair: {pair_counts.mean():.2f}")

print("\nPesaran CD statistic:")
print(f"CD = {CD:.4f}")
print(f"p-value = {p_value:.6f}")

In [ ]:
# ============================================================
# STEP 38 — INSPECT EXTREME PAIRWISE RESIDUAL CORRELATIONS
# ============================================================

pair_results = []

for i in range(len(countries)):
    for j in range(i + 1, len(countries)):

        c1 = countries[i]
        c2 = countries[j]

        pair = resid_wide[[c1, c2]].dropna()

        if len(pair) >= 3:
            r = pair.iloc[:, 0].corr(pair.iloc[:, 1])

            if pd.notna(r):
                pair_results.append({
                    "Country_1": c1,
                    "Country_2": c2,
                    "Correlation": r,
                    "Overlap_Years": len(pair)
                })

pair_results_df = pd.DataFrame(pair_results)

print("="*90)
print("10 MOST POSITIVE PAIRWISE RESIDUAL CORRELATIONS")
print("="*90)

print(
    pair_results_df
    .sort_values("Correlation", ascending=False)
    .head(10)
    .round(4)
    .to_string(index=False)
)

print("\n")
print("="*90)
print("10 MOST NEGATIVE PAIRWISE RESIDUAL CORRELATIONS")
print("="*90)

print(
    pair_results_df
    .sort_values("Correlation", ascending=True)
    .head(10)
    .round(4)
    .to_string(index=False)
)

In [ ]:
# ============================================================
# STEP 38B — DISTRIBUTION OF PAIRWISE CORRELATIONS
# ============================================================

print("="*80)
print("PAIRWISE RESIDUAL CORRELATION DISTRIBUTION")
print("="*80)

print(
    pair_results_df["Correlation"]
    .describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    .round(4)
)

print("\nShare of country pairs by absolute correlation:")

for threshold in [0.10, 0.20, 0.30, 0.50, 0.70, 0.90]:
    share = (
        (pair_results_df["Correlation"].abs() >= threshold).mean() * 100
    )
    print(f"|r| >= {threshold:.2f}: {share:.2f}%")

Test for serial correlation

In [ ]:
# ============================================================
# STEP 39 — WITHIN-COUNTRY RESIDUAL SERIAL CORRELATION
# ============================================================

resid_sorted = residual_df.sort_values(
    ["Country", "Year"]
).copy()

country_acf = []

for country, g in resid_sorted.groupby("Country"):

    g = g.sort_values("Year")

    if len(g) >= 3:
        r1 = g["Residual"].autocorr(lag=1)

        country_acf.append({
            "Country": country,
            "AR1_Residual": r1,
            "Observations": len(g)
        })

acf_df = pd.DataFrame(country_acf)

print("="*90)
print("COUNTRY-LEVEL LAG-1 RESIDUAL AUTOCORRELATION")
print("="*90)

print(
    acf_df["AR1_Residual"]
    .describe(
        percentiles=[
            0.01, 0.05, 0.10, 0.25,
            0.50, 0.75, 0.90, 0.95, 0.99
        ]
    )
    .round(4)
)

print("\nShare of countries:")
for threshold in [0.10, 0.20, 0.30, 0.50]:
    positive = (acf_df["AR1_Residual"] >= threshold).mean() * 100
    negative = (acf_df["AR1_Residual"] <= -threshold).mean() * 100
    absolute = (acf_df["AR1_Residual"].abs() >= threshold).mean() * 100

    print(
        f"|AR(1)| >= {threshold:.2f}: {absolute:.2f}% "
        f"(positive >= {threshold:.2f}: {positive:.2f}%; "
        f"negative <= -{threshold:.2f}: {negative:.2f}%)"
    )

In [ ]:
# ============================================================
# STEP 39B — MEAN AND MEDIAN AR(1)
# ============================================================

print("="*80)
print("SUMMARY OF COUNTRY-LEVEL RESIDUAL AR(1)")
print("="*80)

print(f"Mean AR(1):   {acf_df['AR1_Residual'].mean():.4f}")
print(f"Median AR(1): {acf_df['AR1_Residual'].median():.4f}")
print(f"Std. Dev.:    {acf_df['AR1_Residual'].std():.4f}")

print("\nHighest positive AR(1):")
print(
    acf_df
    .sort_values("AR1_Residual", ascending=False)
    .head(10)
    .round(4)
    .to_string(index=False)
)

print("\nMost negative AR(1):")
print(
    acf_df
    .sort_values("AR1_Residual", ascending=True)
    .head(10)
    .round(4)
    .to_string(index=False)
)

Formal serial-correlation test

In [ ]:
# ============================================================
# STEP 40 — WOOLDRIDGE-STYLE TEST FOR PANEL SERIAL CORRELATION
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm

# Start from the two-way FE residuals
w = residual_df.sort_values(["Country", "Year"]).copy()

# First differences of residuals
w["d_resid"] = w.groupby("Country")["Residual"].diff()

# Lagged residual
w["lag_resid"] = w.groupby("Country")["Residual"].shift(1)

# Keep valid observations
test_data = w.dropna(subset=["d_resid", "lag_resid"]).copy()

# Regression:
# Δe_it = alpha + rho * e_i,t-1 + error
X_test = sm.add_constant(test_data["lag_resid"])
y_test = test_data["d_resid"]

aux_model = sm.OLS(y_test, X_test).fit(
    cov_type="HC1"
)

print("="*80)
print("WOOLDRIDGE-STYLE SERIAL CORRELATION DIAGNOSTIC")
print("="*80)

print(aux_model.summary())

In [ ]:
# ============================================================
# STEP 40B — EXTRACT SERIAL CORRELATION TEST STATISTICS
# ============================================================

coef = aux_model.params["lag_resid"]
p_value = aux_model.pvalues["lag_resid"]
t_stat = aux_model.tvalues["lag_resid"]

print("="*80)
print("SERIAL CORRELATION TEST RESULT")
print("="*80)

print(f"Lagged residual coefficient: {coef:.6f}")
print(f"t-statistic:                 {t_stat:.4f}")
print(f"p-value:                     {p_value:.8f}")

if p_value < 0.05:
    print("\nConclusion: Evidence of serial correlation (p < 0.05).")
else:
    print("\nConclusion: No evidence of serial correlation (p >= 0.05).")

Heteroskedasticity

In [ ]:
# ============================================================
# STEP 41 — COUNTRY-LEVEL RESIDUAL VARIANCE
# ============================================================

resid_variance = (
    residual_df
    .groupby("Country")["Residual"]
    .agg(
        Residual_SD="std",
        Residual_Variance="var",
        Mean_Absolute_Residual=lambda x: np.mean(np.abs(x)),
        N="count"
    )
    .reset_index()
)

print("="*100)
print("COUNTRY-LEVEL RESIDUAL VARIANCE")
print("="*100)

print(
    resid_variance[
        ["Residual_SD", "Residual_Variance",
         "Mean_Absolute_Residual", "N"]
    ]
    .describe()
    .round(4)
)

In [ ]:
# ============================================================
# STEP 41B — RESIDUAL VARIANCE RANGE
# ============================================================

print("="*80)
print("COUNTRIES WITH HIGHEST RESIDUAL VARIANCE")
print("="*80)

print(
    resid_variance
    .sort_values("Residual_Variance", ascending=False)
    .head(15)
    .round(4)
    .to_string(index=False)
)

print("\n")
print("="*80)
print("COUNTRIES WITH LOWEST RESIDUAL VARIANCE")
print("="*80)

print(
    resid_variance
    .sort_values("Residual_Variance", ascending=True)
    .head(15)
    .round(4)
    .to_string(index=False)
)

In [ ]:
# ============================================================
# STEP 41C — RESIDUAL-SCALE DIAGNOSTIC
# ============================================================

from scipy.stats import levene

# Divide countries into quartiles based on their residual variance
resid_variance["Variance_Group"] = pd.qcut(
    resid_variance["Residual_Variance"],
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"],
    duplicates="drop"
)

groups = []

for group in ["Q1", "Q2", "Q3", "Q4"]:
    country_list = resid_variance.loc[
        resid_variance["Variance_Group"] == group,
        "Country"
    ]

    values = residual_df[
        residual_df["Country"].isin(country_list)
    ]["Residual"].values

    groups.append(values)

levene_stat, levene_p = levene(*groups, center="median")

print("="*80)
print("LEVENE TEST FOR DIFFERENCES IN RESIDUAL VARIANCE")
print("="*80)

print(f"Levene statistic = {levene_stat:.4f}")
print(f"p-value = {levene_p:.8f}")

if levene_p < 0.05:
    print("\nConclusion: Evidence of unequal residual variances.")
else:
    print("\nConclusion: No evidence of unequal residual variances.")

Formal panel diagnostics + covariance comparison

Rather than doing another ad hoc test, we're now going to compare several defensible covariance estimators on the exact same two-way FE model.

This will allow us to see whether our substantive conclusions are stable when we account for the error structure.

In [ ]:
# ============================================================
# STEP 42A — COVARIANCE ESTIMATOR COMPARISON
# ============================================================

from linearmodels.panel import PanelOLS

# Re-create indexed dataset
fe_data = model.copy()
fe_data = fe_data.set_index(["Country", "Year"])

y = fe_data["ln_DMC_GDP"]

X = fe_data[
    [
        "asinh_NRR",
        "ln_GDPpc",
        "Industry_level",
        "ln_Trade",
        "Urban_level",
        "Renewable_level",
        "FDI_level"
    ]
]

fe_model = PanelOLS(
    y,
    X,
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)

# ------------------------------------------------------------
# A. Unadjusted
# ------------------------------------------------------------

res_unadjusted = fe_model.fit(
    cov_type="unadjusted"
)

# ------------------------------------------------------------
# B. Country clustered
# ------------------------------------------------------------

res_country = fe_model.fit(
    cov_type="clustered",
    cluster_entity=True
)

# ------------------------------------------------------------
# C. Driscoll-Kraay
# ------------------------------------------------------------

res_dk = fe_model.fit(
    cov_type="kernel",
    kernel="bartlett",
    bandwidth=3
)

# ------------------------------------------------------------
# D. Two-way clustering
# ------------------------------------------------------------

res_tw = fe_model.fit(
    cov_type="clustered",
    cluster_entity=True,
    cluster_time=True
)

In [ ]:
# ============================================================
# STEP 42A-VERIFY — CHECK COVARIANCE ESTIMATIONS
# ============================================================

print("="*80)
print("COVARIANCE ESTIMATOR ESTIMATION CHECK")
print("="*80)

results_dict = {
    "Unadjusted": res_unadjusted,
    "Country-clustered": res_country,
    "Driscoll-Kraay": res_dk,
    "Two-way clustered": res_tw
}

for name, result in results_dict.items():
    print(f"{name:20s} | "
          f"N = {result.nobs} | "
          f"Covariance = {result._cov_type}")

In [ ]:
# ============================================================
# STEP 42B — COMPARISON TABLE
# ============================================================

variables = [
    "asinh_NRR",
    "ln_GDPpc",
    "Industry_level",
    "ln_Trade",
    "Urban_level",
    "Renewable_level",
    "FDI_level"
]

rows = []

results_dict = {
    "Unadjusted": res_unadjusted,
    "Country-clustered": res_country,
    "Driscoll-Kraay": res_dk,
    "Two-way clustered": res_tw
}

for var in variables:
    for name, result in results_dict.items():
        rows.append({
            "Variable": var,
            "SE_Method": name,
            "Coefficient": result.params[var],
            "Std_Error": result.std_errors[var],
            "t_stat": result.tstats[var],
            "p_value": result.pvalues[var]
        })

comparison = pd.DataFrame(rows)

print("="*110)
print("COVARIANCE ESTIMATOR COMPARISON")
print("="*110)

print(
    comparison
    .round(5)
    .to_string(index=False)
)

In [ ]:
# ============================================================
# STEP 42C — FOCUS ON NRR
# ============================================================

print("="*80)
print("NATURAL-RESOURCE RENT COEFFICIENT ACROSS COVARIANCE METHODS")
print("="*80)

print(
    comparison[
        comparison["Variable"] == "asinh_NRR"
    ]
    .round(6)
    .to_string(index=False)
)

Driscoll–Kraay bandwidth sensitivity

In [ ]:
# ============================================================
# STEP 43 — DRISCOLL–KRAAY BANDWIDTH SENSITIVITY
# ============================================================

bandwidths = [1, 2, 3, 4, 5, 6, 7]

dk_rows = []

for bw in bandwidths:

    result = fe_model.fit(
        cov_type="kernel",
        kernel="bartlett",
        bandwidth=bw
    )

    for var in variables:
        dk_rows.append({
            "Bandwidth": bw,
            "Variable": var,
            "Coefficient": result.params[var],
            "Std_Error": result.std_errors[var],
            "t_stat": result.tstats[var],
            "p_value": result.pvalues[var]
        })

dk_sensitivity = pd.DataFrame(dk_rows)

print("="*110)
print("DRISCOLL–KRAAY BANDWIDTH SENSITIVITY")
print("="*110)

print(
    dk_sensitivity
    .round(5)
    .to_string(index=False)
)

In [ ]:
# ============================================================
# STEP 43B — NRR BANDWIDTH SENSITIVITY
# ============================================================

print("="*90)
print("NRR — DRISCOLL–KRAAY BANDWIDTH SENSITIVITY")
print("="*90)

print(
    dk_sensitivity[
        dk_sensitivity["Variable"] == "asinh_NRR"
    ]
    .round(6)
    .to_string(index=False)
)

In [ ]:
# ============================================================
# STEP 43C — KEY VARIABLES
# ============================================================

key_vars = [
    "asinh_NRR",
    "ln_GDPpc",
    "ln_Trade",
    "Renewable_level"
]

print("="*110)
print("KEY VARIABLES — DRISCOLL–KRAAY BANDWIDTH SENSITIVITY")
print("="*110)

print(
    dk_sensitivity[
        dk_sensitivity["Variable"].isin(key_vars)
    ]
    .round(5)
    .to_string(index=False)
)

Formal serial-correlation diagnostic

In [ ]:
# ============================================================
# STEP 44A — CHECK AVAILABLE PANEL DIAGNOSTIC PACKAGES
# ============================================================

import sys
import importlib.util

packages = [
    "linearmodels",
    "statsmodels",
    "arch",
    "pandas",
    "numpy"
]

print("=" * 80)
print("AVAILABLE STATISTICAL PACKAGES")
print("=" * 80)

for pkg in packages:
    available = importlib.util.find_spec(pkg) is not None
    print(f"{pkg:<20} : {'AVAILABLE' if available else 'NOT AVAILABLE'}")

In [ ]:
# ============================================================
# STEP 44B — CHECK LINEARMODELS VERSION AND PANEL TEST OPTIONS
# ============================================================

import linearmodels
import inspect

print("linearmodels version:", linearmodels.__version__)

print("\nPanelOLS available:", hasattr(linearmodels.panel, "PanelOLS"))

print("\nAvailable objects in linearmodels.panel:")
print([x for x in dir(linearmodels.panel) if not x.startswith("_")])

In [ ]:
# ============================================================
# STEP 44C — RESIDUAL SERIAL-CORRELATION SUMMARY
# ============================================================

import numpy as np
import pandas as pd

# Residuals from the two-way FE model
resid = fe_model.fit().resids.copy()

# Convert to dataframe
resid_df = resid.reset_index()

print("=" * 80)
print("PANEL RESIDUAL SERIAL-CORRELATION DIAGNOSTIC")
print("=" * 80)

print("Observations:", len(resid_df))
print("Countries:", resid_df.iloc[:, 0].nunique())
print("Years:", resid_df.iloc[:, 1].nunique())

# Identify index columns
country_col = resid_df.columns[0]
year_col = resid_df.columns[1]
resid_col = resid_df.columns[-1]

# Sort properly
resid_df = resid_df.sort_values([country_col, year_col])

# Lag residual within country
resid_df["lag_resid"] = resid_df.groupby(country_col)[resid_col].shift(1)

# Country-level AR(1)
ar1_rows = []

for country, g in resid_df.groupby(country_col):
    g = g.dropna(subset=["lag_resid", resid_col])

    if len(g) >= 3:
        corr = g[["lag_resid", resid_col]].corr().iloc[0, 1]

        ar1_rows.append({
            "Country": country,
            "N_pairs": len(g),
            "AR1": corr
        })

ar1_df = pd.DataFrame(ar1_rows)

print("\nCountry-level residual AR(1):")
print(ar1_df["AR1"].describe().round(4))

print("\nShare with positive AR(1):",
      round((ar1_df["AR1"] > 0).mean(), 4))

print("Share with AR(1) >= 0.5:",
      round((ar1_df["AR1"] >= 0.5).mean(), 4))

print("Share with AR(1) >= 0.7:",
      round((ar1_df["AR1"] >= 0.7).mean(), 4))

print("\nLowest 10:")
print(ar1_df.nsmallest(10, "AR1").to_string(index=False))

print("\nHighest 10:")
print(ar1_df.nlargest(10, "AR1").to_string(index=False))

Influence and outlier diagnostics

Country-level influence

In [ ]:
# ============================================================
# STEP 45A — COUNTRY-LEVEL LEAVE-ONE-OUT INFLUENCE
# ============================================================

import numpy as np
import pandas as pd
from linearmodels.panel import PanelOLS

print("=" * 90)
print("STEP 45A — COUNTRY-LEVEL LEAVE-ONE-OUT INFLUENCE")
print("=" * 90)

# Work from the final positive-DMC baseline dataset
influence_data = model.copy()

# Set panel index
influence_data = influence_data.set_index(["Country", "Year"])

# Variables
y = influence_data["ln_DMC_GDP"]

X_vars = [
    "asinh_NRR",
    "ln_GDPpc",
    "Industry_level",
    "ln_Trade",
    "Urban_level",
    "Renewable_level",
    "FDI_level"
]

X = influence_data[X_vars]

# Full-sample benchmark
full_result = PanelOLS(
    y, X,
    entity_effects=True,
    time_effects=True
).fit(
    cov_type="kernel",
    kernel="bartlett",
    bandwidth=3
)

full_beta = full_result.params["asinh_NRR"]

print(f"\nFull-sample NRR coefficient: {full_beta:.8f}")

countries = influence_data.index.get_level_values("Country").unique()

loo_rows = []

for country in countries:

    sub = influence_data[
        influence_data.index.get_level_values("Country") != country
    ]

    try:
        res = PanelOLS(
            sub["ln_DMC_GDP"],
            sub[X_vars],
            entity_effects=True,
            time_effects=True
        ).fit(
            cov_type="kernel",
            kernel="bartlett",
            bandwidth=3
        )

        beta = res.params["asinh_NRR"]

        loo_rows.append({
            "Country": country,
            "NRR_beta_leaveout": beta,
            "Difference_from_full": beta - full_beta,
            "Absolute_change": abs(beta - full_beta)
        })

    except Exception as e:
        print("Failed:", country, str(e))

loo_df = pd.DataFrame(loo_rows)

print("\nCountries successfully estimated:", len(loo_df))

print("\nLargest absolute changes:")
print(
    loo_df
    .sort_values("Absolute_change", ascending=False)
    .head(20)
    .round(8)
    .to_string(index=False)
)

print("\nDistribution of leave-one-out NRR coefficients:")
print(
    loo_df["NRR_beta_leaveout"]
    .describe()
    .round(8)
)

print("\nFull-sample beta:", round(full_beta, 8))
print("Minimum LOO beta:", round(loo_df["NRR_beta_leaveout"].min(), 8))
print("Maximum LOO beta:", round(loo_df["NRR_beta_leaveout"].max(), 8))

Specifically test the known unusual observations

In [ ]:
# ============================================================
# STEP 45B — SENSITIVITY TO SOURCE-NATIVE NEGATIVE DMC VALUES
# ============================================================

print("=" * 90)
print("STEP 45B — NEGATIVE DMC/GDP OBSERVATION SENSITIVITY")
print("=" * 90)

negative_obs = [
    ("Brunei Darussalam", 2020),
    ("Brunei Darussalam", 2021),
    ("Mozambique", 2016),
    ("Mozambique", 2017)
]

data_no_negative = model.copy()

for country, year in negative_obs:
    data_no_negative = data_no_negative[
        ~(
            (data_no_negative["Country"] == country) &
            (data_no_negative["Year"] == year)
        )
    ]

data_no_negative = data_no_negative.set_index(["Country", "Year"])

result_no_negative = PanelOLS(
    data_no_negative["ln_DMC_GDP"],
    data_no_negative[X_vars],
    entity_effects=True,
    time_effects=True
).fit(
    cov_type="kernel",
    kernel="bartlett",
    bandwidth=3
)

beta_no_negative = result_no_negative.params["asinh_NRR"]

print("Full sample N:", len(influence_data))
print("Without four negative-DMC observations N:", len(data_no_negative))

print("\nFull-sample NRR coefficient:",
      round(full_beta, 8))

print("Without negative-DMC observations:",
      round(beta_no_negative, 8))

print("Change:",
      round(beta_no_negative - full_beta, 8))

print("\nFull coefficient table:")
print(
    pd.DataFrame({
        "Full": full_result.params,
        "Without_negative_DMC": result_no_negative.params
    }).round(8)
)

FDI influence sensitivity

Because FDI is extremely skewed, we'll also test the model with asinh-transformed FDI rather than deleting extreme observations.

In [ ]:
# ============================================================
# STEP 45C — FDI TRANSFORMATION SENSITIVITY
# ============================================================

print("=" * 90)
print("STEP 45C — FDI TRANSFORMATION SENSITIVITY")
print("=" * 90)

fdi_test = model.copy()

fdi_test["asinh_FDI"] = np.arcsinh(fdi_test["FDI_level"])

fdi_test = fdi_test.set_index(["Country", "Year"])

X_fdi = fdi_test[
    [
        "asinh_NRR",
        "ln_GDPpc",
        "Industry_level",
        "ln_Trade",
        "Urban_level",
        "Renewable_level",
        "asinh_FDI"
    ]
]

fdi_result = PanelOLS(
    fdi_test["ln_DMC_GDP"],
    X_fdi,
    entity_effects=True,
    time_effects=True
).fit(
    cov_type="kernel",
    kernel="bartlett",
    bandwidth=3
)

print(fdi_result.summary)

print("\nNRR coefficient with asinh(FDI):",
      round(fdi_result.params["asinh_NRR"], 8))

print("NRR p-value:",
      round(fdi_result.pvalues["asinh_NRR"], 8))

First, check the resource components

In [ ]:
# ============================================================
# STEP 46A — RESOURCE COMPONENT WITHIN VARIATION
# ============================================================

import numpy as np
import pandas as pd

print("=" * 90)
print("STEP 46A — RESOURCE COMPONENT WITHIN-COUNTRY VARIATION")
print("=" * 90)

resource_vars = [
    "Oil",
    "Gas",
    "Coal",
    "Mineral",
    "Forest"
]

resource_check = []

for var in resource_vars:

    # Country means
    country_mean = model.groupby("Country")[var].transform("mean")

    # Within-country deviations
    within = model[var] - country_mean

    # Country-specific within SD
    country_within_sd = within.groupby(model["Country"]).std()

    resource_check.append({
        "Variable": var,
        "Pooled_SD": model[var].std(),
        "Within_SD": within.std(),
        "Within_to_Pooled": within.std() / model[var].std(),
        "Countries_with_variation": (country_within_sd > 0).sum(),
        "Countries_zero_within_variation": (country_within_sd == 0).sum(),
        "Median_country_within_SD": country_within_sd.median(),
        "Max_country_within_SD": country_within_sd.max()
    })

resource_variation = pd.DataFrame(resource_check)

print(resource_variation.round(4).to_string(index=False))

In [ ]:
# ============================================================
# STEP 46B — POOLED AND WITHIN CORRELATIONS
# ============================================================

print("=" * 90)
print("STEP 46B — RESOURCE COMPONENT CORRELATIONS")
print("=" * 90)

print("\nPOOLED CORRELATIONS")
print(
    model[resource_vars]
    .corr()
    .round(3)
    .to_string()
)

# Within-country demeaned values
within_resource = model[resource_vars].copy()

for var in resource_vars:
    within_resource[var] = (
        model[var]
        - model.groupby("Country")[var].transform("mean")
    )

print("\nWITHIN-COUNTRY CORRELATIONS")
print(
    within_resource
    .corr()
    .round(3)
    .to_string()
)

In [ ]:
# ============================================================
# STEP 46C — WITHIN-VIF FOR RESOURCE COMPONENTS
# ============================================================

from statsmodels.stats.outliers_influence import variance_inflation_factor

print("=" * 90)
print("STEP 46C — WITHIN-COUNTRY VIF: RESOURCE COMPONENT MODEL")
print("=" * 90)

vif_data = within_resource.copy()

vif_data = vif_data.dropna()

vif_rows = []

for i, var in enumerate(resource_vars):

    vif_rows.append({
        "Variable": var,
        "Within_VIF": variance_inflation_factor(
            vif_data.values,
            i
        )
    })

resource_vif = pd.DataFrame(vif_rows)

print(resource_vif.round(3).to_string(index=False))

Distribution and zero-pattern audit


In [ ]:
# ============================================================
# STEP 46D — RESOURCE RENT DISTRIBUTION AUDIT
# ============================================================

print("=" * 90)
print("STEP 46D — RESOURCE RENT DISTRIBUTION AUDIT")
print("=" * 90)

resource_vars = [
    "Oil",
    "Gas",
    "Coal",
    "Mineral",
    "Forest"
]

distribution_rows = []

for var in resource_vars:

    s = model[var].dropna()

    distribution_rows.append({
        "Variable": var,
        "N": len(s),
        "Mean": s.mean(),
        "Median": s.median(),
        "SD": s.std(),
        "Min": s.min(),
        "P75": s.quantile(.75),
        "P90": s.quantile(.90),
        "P95": s.quantile(.95),
        "P99": s.quantile(.99),
        "Max": s.max(),
        "Skewness": s.skew(),
        "Zero_N": (s == 0).sum(),
        "Zero_%": 100 * (s == 0).mean(),
        "Positive_N": (s > 0).sum()
    })

resource_distribution = pd.DataFrame(distribution_rows)

print(resource_distribution.round(4).to_string(index=False))

In [ ]:
# ============================================================
# STEP 46E — RESOURCE RENT EXTREME OBSERVATIONS
# ============================================================

print("=" * 90)
print("STEP 46E — RESOURCE RENT UPPER-TAIL OBSERVATIONS")
print("=" * 90)

for var in resource_vars:

    print("\n" + "-" * 70)
    print(var)

    print(
        model[["Country", "Year", var]]
        .dropna()
        .sort_values(var, ascending=False)
        .head(10)
        .to_string(index=False)
    )

In [ ]:
# ============================================================
# STEP 46F — RESOURCE COMPONENT SUM VS TOTAL NRR
# ============================================================

print("=" * 90)
print("STEP 46F — RESOURCE COMPONENT ACCOUNTING CHECK")
print("=" * 90)

accounting = model.copy()

accounting["Component_Sum"] = accounting[
    ["Oil", "Gas", "Coal", "Mineral", "Forest"]
].sum(axis=1, min_count=5)

accounting["NRR_Difference"] = (
    accounting["NRR"] - accounting["Component_Sum"]
)

check = accounting["NRR_Difference"].dropna()

print("\nN observations with all components:", len(check))

print("\nDifference statistics:")
print(check.describe().round(6))

print("\nAbsolute difference statistics:")
print(check.abs().describe().round(6))

print("\nLargest absolute differences:")
print(
    accounting[
        ["Country", "Year", "NRR",
         "Oil", "Gas", "Coal", "Mineral", "Forest",
         "Component_Sum", "NRR_Difference"]
    ]
    .dropna()
    .assign(Abs_Difference=lambda x: x["NRR_Difference"].abs())
    .sort_values("Abs_Difference", ascending=False)
    .head(20)
    .round(6)
    .to_string(index=False)
)

Composition-model sample audit

In [ ]:
# ============================================================
# STEP 47A — FIVE-RESOURCE MODEL SAMPLE AUDIT
# CORRECTED VERSION
# ============================================================

print("=" * 90)
print("STEP 47A — FIVE-RESOURCE MODEL SAMPLE AUDIT")
print("=" * 90)

resource_vars = [
    "Oil",
    "Gas",
    "Coal",
    "Mineral",
    "Forest"
]

composition_vars = [
    "Country",
    "Year",
    "ln_DMC_GDP",
    "Oil",
    "Gas",
    "Coal",
    "Mineral",
    "Forest",
    "ln_GDPpc",
    "Industry_level",
    "ln_Trade",
    "Urban_level",
    "Renewable_level",
    "FDI_level"
]

# Complete-case composition sample
composition_data = model[composition_vars].dropna().copy()

print("\nBaseline observations:", len(model))
print("Composition complete cases:", len(composition_data))
print("Observations lost:", len(model) - len(composition_data))
print(
    "Percentage retained:",
    round(100 * len(composition_data) / len(model), 2)
)

print(
    "Countries:",
    composition_data["Country"].nunique()
)

print(
    "Years:",
    composition_data["Year"].min(),
    "to",
    composition_data["Year"].max()
)

# Country-year observations per country
country_counts = (
    composition_data
    .groupby("Country")
    .size()
)

print("\nCountry-year observations per country:")
print(country_counts.describe().round(2))

print("\nCountries with fewer than 18 observations:")

fewer18 = country_counts[country_counts < 18].sort_values()

if len(fewer18) == 0:
    print("NONE")
else:
    print(fewer18.to_string())

In [ ]:
# ============================================================
# STEP 47B — WHICH RESOURCE VARIABLES CAUSE THE 87 LOSSES?
# ============================================================

print("=" * 90)
print("STEP 47B — SOURCE OF COMPOSITION-MODEL MISSINGNESS")
print("=" * 90)

for var in resource_vars:

    missing = model[var].isna().sum()

    print(
        f"{var:<10} missing: {missing:>4} "
        f"({100 * missing / len(model):.2f}%)"
    )

# Any missing resource component
missing_resource = model[resource_vars].isna().any(axis=1)

print("\nRows missing at least one resource component:")
print("N =", missing_resource.sum())

print(
    "Countries affected:",
    model.loc[missing_resource, "Country"].nunique()
)

print("\nAffected countries — number of missing country-years:")

affected = (
    model.loc[missing_resource]
    .groupby("Country")
    .size()
    .sort_values(ascending=False)
)

print(affected.to_string())

In [ ]:
# ============================================================
# STEP 47C — COMPOSITION SAMPLE VS BASELINE SAMPLE
# ============================================================

print("=" * 90)
print("STEP 47C — SAMPLE COMPARISON")
print("=" * 90)

baseline_countries = set(model["Country"])
composition_countries = set(composition_data["Country"])

print(
    "Countries in baseline:",
    len(baseline_countries)
)

print(
    "Countries in composition:",
    len(composition_countries)
)

lost_countries = baseline_countries - composition_countries

print(
    "Countries lost:",
    len(lost_countries)
)

if lost_countries:
    print("\nCountries lost from composition model:")
    print(sorted(lost_countries))
else:
    print("\nNo countries completely lost from composition model.")

# Variables for mean comparison
comparison_vars = [
    "ln_DMC_GDP",
    "ln_GDPpc",
    "Industry_level",
    "ln_Trade",
    "Urban_level",
    "Renewable_level",
    "FDI_level"
]

baseline_means = model[comparison_vars].mean()

composition_means = composition_data[comparison_vars].mean()

sample_comparison = pd.DataFrame({
    "Baseline_mean": baseline_means,
    "Composition_mean": composition_means,
    "Difference": composition_means - baseline_means
})

print("\nMean comparison:")
print(sample_comparison.round(4).to_string())

In [ ]:
# ============================================================
# STEP 47D — APPLY THE >=18-YEAR ELIGIBILITY RULE
# ============================================================

print("=" * 90)
print("STEP 47D — COMPOSITION SAMPLE >=18-YEAR ELIGIBILITY")
print("=" * 90)

composition_counts = (
    composition_data
    .groupby("Country")
    .size()
)

eligible_composition_countries = composition_counts[
    composition_counts >= 18
].index

composition_ge18 = composition_data[
    composition_data["Country"].isin(eligible_composition_countries)
].copy()

print(
    "Composition complete-case observations:",
    len(composition_data)
)

print(
    "Composition >=18-year observations:",
    len(composition_ge18)
)

print(
    "Countries with >=18 observations:",
    len(eligible_composition_countries)
)

print(
    "Minimum observations per eligible country:",
    composition_ge18.groupby("Country").size().min()
)

print(
    "Maximum observations per eligible country:",
    composition_ge18.groupby("Country").size().max()
)

print(
    "Mean observations per eligible country:",
    round(
        composition_ge18.groupby("Country").size().mean(),
        2
    )
)

print("\nCountry counts:")
print(
    composition_ge18
    .groupby("Country")
    .size()
    .describe()
    .round(2)
)

Now estimate the actual resource-type model

In [ ]:
# ============================================================
# STEP 48A — FIVE-RESOURCE TWO-WAY FE MODEL
# ============================================================

import numpy as np
import pandas as pd
from linearmodels.panel import PanelOLS

print("=" * 100)
print("STEP 48A — FIVE-RESOURCE TWO-WAY FIXED-EFFECTS MODEL")
print("=" * 100)

# Use the >=18-year eligible composition sample
comp = composition_ge18.copy()

# Set panel index
comp = comp.set_index(["Country", "Year"])

# Dependent variable
y_comp = comp["ln_DMC_GDP"]

# Resource components + same controls as baseline
resource_model_vars = [
    "Oil",
    "Gas",
    "Coal",
    "Mineral",
    "Forest",
    "ln_GDPpc",
    "Industry_level",
    "ln_Trade",
    "Urban_level",
    "Renewable_level",
    "FDI_level"
]

X_comp = comp[resource_model_vars]

# Two-way FE + Driscoll-Kraay
resource_fe = PanelOLS(
    y_comp,
    X_comp,
    entity_effects=True,
    time_effects=True
)

resource_result = resource_fe.fit(
    cov_type="kernel",
    kernel="bartlett",
    bandwidth=3
)

print(resource_result.summary)

In [ ]:
# ============================================================
# STEP 48B — RESOURCE MODEL COEFFICIENT TABLE
# ============================================================

resource_table = pd.DataFrame({
    "Coefficient": resource_result.params,
    "Std_Error": resource_result.std_errors,
    "t_stat": resource_result.tstats,
    "p_value": resource_result.pvalues,
    "CI_Lower": resource_result.conf_int().iloc[:, 0],
    "CI_Upper": resource_result.conf_int().iloc[:, 1]
})

print("\nRESOURCE-TYPE MODEL — COEFFICIENT TABLE")
print("=" * 100)

print(
    resource_table
    .loc[resource_model_vars]
    .round(6)
    .to_string()
)

In [ ]:
# ============================================================
# STEP 48C — RESOURCE MODEL SAMPLE INFORMATION
# ============================================================

print("\n" + "=" * 100)
print("RESOURCE MODEL SAMPLE")
print("=" * 100)

print("Observations:", resource_result.nobs)
print("Countries:", comp.index.get_level_values("Country").nunique())
print("Years:", comp.index.get_level_values("Year").nunique())

country_n = comp.groupby(level="Country").size()

print("Minimum observations/country:", country_n.min())
print("Maximum observations/country:", country_n.max())
print("Mean observations/country:", round(country_n.mean(), 2))
print("Median observations/country:", country_n.median())

Resource-model robustness

In [ ]:
# ============================================================
# STEP 49A — RESOURCE MODEL DK BANDWIDTH SENSITIVITY
# ============================================================

bandwidths = [1, 2, 3, 4, 5, 6, 7]

resource_dk_rows = []

for bw in bandwidths:

    result = resource_fe.fit(
        cov_type="kernel",
        kernel="bartlett",
        bandwidth=bw
    )

    for var in resource_model_vars:

        resource_dk_rows.append({
            "Bandwidth": bw,
            "Variable": var,
            "Coefficient": result.params[var],
            "Std_Error": result.std_errors[var],
            "t_stat": result.tstats[var],
            "p_value": result.pvalues[var]
        })

resource_dk_sensitivity = pd.DataFrame(resource_dk_rows)

print("=" * 100)
print("RESOURCE MODEL — DRISCOLL–KRAAY BANDWIDTH SENSITIVITY")
print("=" * 100)

print(
    resource_dk_sensitivity
    .round(6)
    .to_string(index=False)
)

In [ ]:
# ============================================================
# STEP 49B — RESOURCE VARIABLES ONLY
# ============================================================

resource_only = resource_dk_sensitivity[
    resource_dk_sensitivity["Variable"].isin(
        ["Oil", "Gas", "Coal", "Mineral", "Forest"]
    )
]

print("=" * 100)
print("RESOURCE VARIABLES — DK BANDWIDTH SENSITIVITY")
print("=" * 100)

print(
    resource_only
    .round(6)
    .to_string(index=False)
)

In [ ]:
# ============================================================
# STEP 49C — PAIRWISE RESOURCE COEFFICIENT EQUALITY TESTS
# ============================================================

print("=" * 100)
print("STEP 49C — PAIRWISE RESOURCE COEFFICIENT EQUALITY TESTS")
print("=" * 100)

pairs = [
    ("Oil", "Gas"),
    ("Oil", "Coal"),
    ("Oil", "Mineral"),
    ("Oil", "Forest"),
    ("Gas", "Coal"),
    ("Gas", "Mineral"),
    ("Gas", "Forest"),
    ("Coal", "Mineral"),
    ("Coal", "Forest"),
    ("Mineral", "Forest")
]

equality_results = []

for v1, v2 in pairs:

    restriction = f"{v1} = {v2}"

    try:
        test = resource_result.wald_test(formula=restriction)

        equality_results.append({
            "Comparison": f"{v1} = {v2}",
            "Statistic": float(test.stat),
            "P_value": float(test.pval)
        })

    except Exception as e:

        print(
            f"Could not test {v1} = {v2}: {e}"
        )

equality_df = pd.DataFrame(equality_results)

print(
    equality_df
    .round(6)
    .to_string(index=False)
)

Resource-rent tail sensitivity

In [ ]:
# ============================================================
# STEP 50A — COUNTRY-LEVEL LEAVE-ONE-OUT
# RESOURCE COMPOSITION MODEL
# ============================================================

print("=" * 100)
print("STEP 50A — COUNTRY-LEVEL LEAVE-ONE-OUT: RESOURCE MODEL")
print("=" * 100)

full_resource_beta = resource_result.params[
    ["Oil", "Gas", "Coal", "Mineral", "Forest"]
]

countries = comp.index.get_level_values("Country").unique()

resource_loo_rows = []

for country in countries:

    sub = comp[
        comp.index.get_level_values("Country") != country
    ]

    try:

        res = PanelOLS(
            sub["ln_DMC_GDP"],
            sub[resource_model_vars],
            entity_effects=True,
            time_effects=True
        ).fit(
            cov_type="kernel",
            kernel="bartlett",
            bandwidth=3
        )

        row = {"Country": country}

        for var in ["Oil", "Gas", "Coal", "Mineral", "Forest"]:

            row[f"{var}_beta"] = res.params[var]
            row[f"{var}_change"] = (
                res.params[var] - full_resource_beta[var]
            )

        resource_loo_rows.append(row)

    except Exception as e:
        print("Failed:", country, str(e))

resource_loo = pd.DataFrame(resource_loo_rows)

print("\nCountries successfully estimated:",
      len(resource_loo))

for var in ["Oil", "Gas", "Coal", "Mineral", "Forest"]:

    print("\n" + "-" * 80)
    print(var)

    print(
        resource_loo[
            [
                "Country",
                f"{var}_beta",
                f"{var}_change"
            ]
        ]
        .assign(
            Abs_Change=lambda x:
            x[f"{var}_change"].abs()
        )
        .sort_values("Abs_Change", ascending=False)
        .head(10)
        .round(6)
        .to_string(index=False)
    )

    print(
        "\nFull coefficient:",
        round(full_resource_beta[var], 6)
    )

    print(
        "LOO minimum:",
        round(resource_loo[f"{var}_beta"].min(), 6)
    )

    print(
        "LOO maximum:",
        round(resource_loo[f"{var}_beta"].max(), 6)
    )

Asinh resource-tail sensitivity

In [ ]:
import numpy as np
from linearmodels.panel import PanelOLS

# Copy the composition sample
comp_asinh = comp.copy()

# Asinh transformations preserve legitimate zeros
for var in ["Oil", "Gas", "Coal", "Mineral", "Forest"]:
    comp_asinh[f"asinh_{var}"] = np.arcsinh(comp_asinh[var])

asinh_resource_vars = [
    "asinh_Oil",
    "asinh_Gas",
    "asinh_Coal",
    "asinh_Mineral",
    "asinh_Forest",
    "ln_GDPpc",
    "Industry_level",
    "ln_Trade",
    "Urban_level",
    "Renewable_level",
    "FDI_level"
]

# Estimate two-way FE with DK SE, bandwidth 3
asinh_result = PanelOLS(
    comp_asinh["ln_DMC_GDP"],
    comp_asinh[asinh_resource_vars],
    entity_effects=True,
    time_effects=True
).fit(
    cov_type="kernel",
    kernel="bartlett",
    bandwidth=3
)

print("="*100)
print("STEP 50B — ASINH RESOURCE-TAIL SENSITIVITY")
print("="*100)

print(asinh_result.summary)

print("\nRESOURCE COEFFICIENTS")
print("-"*80)

for var in ["asinh_Oil", "asinh_Gas", "asinh_Coal",
            "asinh_Mineral", "asinh_Forest"]:
    print(
        f"{var:18s} "
        f"beta={asinh_result.params[var]: .6f}   "
        f"SE={asinh_result.std_errors[var]: .6f}   "
        f"t={asinh_result.tstats[var]: .4f}   "
        f"p={asinh_result.pvalues[var]: .6f}"
    )

In [ ]:
# ============================================================
# STEP 50C — SAME-SAMPLE AGGREGATE NRR MODEL
# Robust version
# ============================================================

same_sample = comp.copy()

print("="*100)
print("STEP 50C — CHECKING COMPOSITION DATAFRAME")
print("="*100)

print("\nColumns available in comp:")
print(list(same_sample.columns))

# ------------------------------------------------------------
# 1. Reconstruct total NRR from its five components
# ------------------------------------------------------------

resource_components = ["Oil", "Gas", "Coal", "Mineral", "Forest"]

missing_components = [
    x for x in resource_components
    if x not in same_sample.columns
]

if missing_components:
    raise KeyError(
        f"These resource columns are missing from comp: {missing_components}"
    )

same_sample["NRR_reconstructed"] = (
    same_sample["Oil"]
    + same_sample["Gas"]
    + same_sample["Coal"]
    + same_sample["Mineral"]
    + same_sample["Forest"]
)

# ------------------------------------------------------------
# 2. Create aggregate NRR transformation
# ------------------------------------------------------------

same_sample["asinh_NRR"] = np.arcsinh(
    same_sample["NRR_reconstructed"]
)

# ------------------------------------------------------------
# 3. Ensure all control variables exist
# ------------------------------------------------------------

if "ln_GDPpc" not in same_sample.columns:
    same_sample["ln_GDPpc"] = np.log(same_sample["GDPpc"])

if "Industry_level" not in same_sample.columns:
    same_sample["Industry_level"] = same_sample["Industry"]

if "ln_Trade" not in same_sample.columns:
    same_sample["ln_Trade"] = np.log(same_sample["Trade"])

if "Urban_level" not in same_sample.columns:
    same_sample["Urban_level"] = same_sample["Urban"]

if "Renewable_level" not in same_sample.columns:
    same_sample["Renewable_level"] = same_sample["Renewable"]

if "FDI_level" not in same_sample.columns:
    same_sample["FDI_level"] = same_sample["FDI"]

# ------------------------------------------------------------
# 4. Variables for aggregate model
# ------------------------------------------------------------

aggregate_vars_same = [
    "asinh_NRR",
    "ln_GDPpc",
    "Industry_level",
    "ln_Trade",
    "Urban_level",
    "Renewable_level",
    "FDI_level"
]

# ------------------------------------------------------------
# 5. Check completeness
# ------------------------------------------------------------

print("\nMissing values in aggregate-model variables:")
print(same_sample[aggregate_vars_same].isna().sum())

print("\nObservations before estimation:", len(same_sample))

# ------------------------------------------------------------
# 6. Estimate two-way FE + Driscoll-Kraay
# ------------------------------------------------------------

aggregate_same_result = PanelOLS(
    same_sample["ln_DMC_GDP"],
    same_sample[aggregate_vars_same],
    entity_effects=True,
    time_effects=True
).fit(
    cov_type="kernel",
    kernel="bartlett",
    bandwidth=3
)

# ------------------------------------------------------------
# 7. Output
# ------------------------------------------------------------

print("\n" + "="*100)
print("STEP 50C — SAME-SAMPLE AGGREGATE NRR MODEL")
print("="*100)

print(aggregate_same_result.summary)

print("\nKEY NRR RESULT")
print("-"*80)

print(
    f"asinh_NRR: "
    f"beta={aggregate_same_result.params['asinh_NRR']:.6f}, "
    f"SE={aggregate_same_result.std_errors['asinh_NRR']:.6f}, "
    f"t={aggregate_same_result.tstats['asinh_NRR']:.4f}, "
    f"p={aggregate_same_result.pvalues['asinh_NRR']:.6f}"
)

print(f"\nObservations: {aggregate_same_result.nobs}")
print(
    f"Countries: "
    f"{same_sample.index.get_level_values('Country').nunique()}"
)

print("\n" + "="*100)

In [ ]:
# ============================================================
# STEP 50D — MATERIAL FOOTPRINT ROBUSTNESS
# GDP/MF as alternative dependent variable
# ============================================================

# Start from the clean master
mf = master.copy()

# Keep 2000–2021
mf = mf[(mf["Year"] >= 2000) & (mf["Year"] <= 2021)].copy()

# GDP/MF must be strictly positive for log transformation
mf = mf[mf["GDP/MF"] > 0].copy()

# Create transformations
mf["ln_GDP_MF"] = np.log(mf["GDP/MF"])
mf["asinh_NRR"] = np.arcsinh(mf["NRR"])
mf["ln_GDPpc"] = np.log(mf["GDPpc"])
mf["Industry_level"] = mf["Industry"]
mf["ln_Trade"] = np.log(mf["Trade"])
mf["Urban_level"] = mf["Urban"]
mf["Renewable_level"] = mf["Renewable"]
mf["FDI_level"] = mf["FDI"]

mf_vars = [
    "ln_GDP_MF",
    "asinh_NRR",
    "ln_GDPpc",
    "Industry_level",
    "ln_Trade",
    "Urban_level",
    "Renewable_level",
    "FDI_level"
]

# Complete cases
mf = mf.dropna(subset=mf_vars).copy()

# ------------------------------------------------------------
# Apply >=18 observations per country rule
# ------------------------------------------------------------

country_counts = mf.groupby("Country")["Year"].count()

eligible_countries = country_counts[country_counts >= 18].index

mf = mf[mf["Country"].isin(eligible_countries)].copy()

# Set panel index
mf = mf.set_index(["Country", "Year"]).sort_index()

print("="*100)
print("STEP 50D — MATERIAL FOOTPRINT ROBUSTNESS")
print("="*100)

print(f"Observations: {len(mf)}")
print(f"Countries: {mf.index.get_level_values('Country').nunique()}")
print(
    f"Years: "
    f"{mf.index.get_level_values('Year').min()}–"
    f"{mf.index.get_level_values('Year').max()}"
)

print("\nObservations per country:")
print(mf.groupby(level="Country").size().describe())

# ------------------------------------------------------------
# Estimate aggregate NRR robustness model
# ------------------------------------------------------------

mf_result = PanelOLS(
    mf["ln_GDP_MF"],
    mf[
        [
            "asinh_NRR",
            "ln_GDPpc",
            "Industry_level",
            "ln_Trade",
            "Urban_level",
            "Renewable_level",
            "FDI_level"
        ]
    ],
    entity_effects=True,
    time_effects=True
).fit(
    cov_type="kernel",
    kernel="bartlett",
    bandwidth=3
)

print("\n" + "="*100)
print("GDP/MF ROBUSTNESS — AGGREGATE NRR")
print("="*100)

print(mf_result.summary)

print("\nKEY RESULT")
print("-"*80)

print(
    f"asinh_NRR: "
    f"beta={mf_result.params['asinh_NRR']:.6f}, "
    f"SE={mf_result.std_errors['asinh_NRR']:.6f}, "
    f"t={mf_result.tstats['asinh_NRR']:.4f}, "
    f"p={mf_result.pvalues['asinh_NRR']:.6f}"
)

In [ ]:
# ============================================================
# FINAL ROBUSTNESS — RESOURCE COMPOSITION USING GDP/MF
# ============================================================

mf_comp = master.copy()

# Common study period
mf_comp = mf_comp[
    (mf_comp["Year"] >= 2000) &
    (mf_comp["Year"] <= 2021)
].copy()

# GDP/MF must be strictly positive for log transformation
mf_comp = mf_comp[mf_comp["GDP/MF"] > 0].copy()

# Transform dependent variable
mf_comp["ln_GDP_MF"] = np.log(mf_comp["GDP/MF"])

# Controls
mf_comp["ln_GDPpc"] = np.log(mf_comp["GDPpc"])
mf_comp["Industry_level"] = mf_comp["Industry"]
mf_comp["ln_Trade"] = np.log(mf_comp["Trade"])
mf_comp["Urban_level"] = mf_comp["Urban"]
mf_comp["Renewable_level"] = mf_comp["Renewable"]
mf_comp["FDI_level"] = mf_comp["FDI"]

# Five resource components
resource_mf_vars = [
    "Oil",
    "Gas",
    "Coal",
    "Mineral",
    "Forest",
    "ln_GDPpc",
    "Industry_level",
    "ln_Trade",
    "Urban_level",
    "Renewable_level",
    "FDI_level"
]

# Complete cases
mf_comp = mf_comp.dropna(
    subset=["ln_GDP_MF"] + resource_mf_vars
).copy()

# Keep countries with >=18 usable years
country_counts = mf_comp.groupby("Country")["Year"].count()

eligible_countries = country_counts[
    country_counts >= 18
].index

mf_comp = mf_comp[
    mf_comp["Country"].isin(eligible_countries)
].copy()

# Panel index
mf_comp = mf_comp.set_index(
    ["Country", "Year"]
).sort_index()

print("="*100)
print("FINAL ROBUSTNESS — RESOURCE COMPOSITION WITH GDP/MF")
print("="*100)

print(f"Observations: {len(mf_comp)}")
print(
    f"Countries: "
    f"{mf_comp.index.get_level_values('Country').nunique()}"
)
print(
    f"Years: "
    f"{mf_comp.index.get_level_values('Year').min()}–"
    f"{mf_comp.index.get_level_values('Year').max()}"
)

print("\nObservations per country:")
print(mf_comp.groupby(level="Country").size().describe())

# ------------------------------------------------------------
# Estimate five-resource model
# ------------------------------------------------------------

mf_resource_result = PanelOLS(
    mf_comp["ln_GDP_MF"],
    mf_comp[resource_mf_vars],
    entity_effects=True,
    time_effects=True
).fit(
    cov_type="kernel",
    kernel="bartlett",
    bandwidth=3
)

print("\n" + "="*100)
print("GDP/MF — RESOURCE COMPOSITION MODEL")
print("="*100)

print(mf_resource_result.summary)

print("\nRESOURCE COEFFICIENTS")
print("-"*80)

for var in ["Oil", "Gas", "Coal", "Mineral", "Forest"]:
    print(
        f"{var:10s} "
        f"beta={mf_resource_result.params[var]: .6f}   "
        f"SE={mf_resource_result.std_errors[var]: .6f}   "
        f"t={mf_resource_result.tstats[var]: .4f}   "
        f"p={mf_resource_result.pvalues[var]: .6f}"
    )

In [ ]:
mf = mf[mf["GDP/MF"] > 0].copy()
mf["ln_GDP_MF"] = np.log(mf["GDP/MF"])